# Lean 9 : Multi-Agents avec Semantic Kernel

**Navigation** : [← Lean-08-Agentic-Proving-Python](Lean-08-Agentic-Proving-Python.ipynb) | [Index](Lean-01-Setup-Lean-Python.ipynb) | [Lean-10-LeanDojo →](Lean-10-LeanDojo.ipynb)

***



## Description du notebook

### Objectifs pedagogiques

Ce notebook explore l'architecture **multi-agents avec Semantic Kernel** pour la demonstration automatique de théorèmes Lean. Vous apprendrez a :

1. Concevoir un système multi-agents avec 5 rôles specialises (Search, Tactic, Critic, Coordinator, Vérification)
2. Utiliser les **plugins Semantic Kernel** pour exposer l'état partagé (`ProofState`) aux agents LLM
3. Implementer des stratégies d'orchestration : sélection d'agent basee sur l'état, detection de terminaison
4. Coordonner les agents de facon autonome pour prouver des théorèmes Lean/Mathlib

### Prerequis

- **Lean-1 a Lean-8** : Installation, syntaxe Lean 4, tactiques de preuve, workflow agentique
- **Python** : Classes, decorateurs, programmation asynchrone (`async`/`await`), générateurs
- **Semantic Kernel** : Notions de base — plugins, fonctions kernel, `@kernel_function`
- **LLM API** : Acces a un modèle (OpenAI, Anthropic ou modèle local compatible)

### Duree estimée

3 a 4 heures (lecture, exécution des 4 demonstrations et exercice)

## 🎯 Architecture du Système Multi-Agents

### Vue d'ensemble

Notre système utilise **5 agents spécialisés** qui collaborent pour prouver des théorèmes Lean :

1. **SearchAgent** : Recherche de lemmes pertinents dans Mathlib
2. **TacticAgent** : Génération de tactiques Lean appropriées
3. **VerifierAgent** : Vérification formelle des preuves
4. **CriticAgent** : Analyse et suggestions d'amélioration
5. **CoordinatorAgent** : Orchestration et décisions stratégiques

### Pourquoi 5 agents ?

Chaque agent a une **responsabilité unique** (principe de séparation des préoccupations) :

- **Séparation des compétences** : Recherche ≠ Génération ≠ Vérification
- **Spécialisation** : Chaque LLM est prompté pour une tâche précise
- **Robustesse** : Si un agent échoue, les autres continuent
- **Traçabilité** : On sait quel agent a pris quelle décision

### Communication : État partagé vs Message passing

Deux approches classiques en multi-agents :

| **Message Passing** | **État Partagé** (notre choix) |
|---------------------|--------------------------------|
| Agents s'envoient des messages | Tous les agents lisent/écrivent un état central |
| Décentralisé | Centralisé |
| Complexe à orchestrer | Facile à suivre |
| Pas de snapshot global | Snapshot complet à chaque itération |

**Pourquoi état partagé ?**

- Besoin de **cohérence globale** (historique des tactiques, métriques)
- **Debugging facilité** : On peut inspecter l'état après chaque tour
- **Snapshots JSON** : Permet de reproduire exactement une session
- Semantic Kernel supporte ce pattern avec les **plugins**

## 1. Introduction : Semantic Kernel pour Preuves (Python)

### Vue d'ensemble

Microsoft **Semantic Kernel** est un SDK qui permet d'orchestrer des LLMs avec des plugins, de la mémoire et des agents intelligents. Nous allons implementer un système multi-agents pour theorem proving inspire des patterns utilises dans l'analyse argumentative (voir `Argument_Analysis` notebooks).

**Composants cles** :
- **Kernel** : Point d'entrée principal, configure les services LLM
- **Plugins** : Fonctions appelables par les agents (decorated avec `@kernel_function`)
- **Agents** : Entites autonomes avec instructions et capacites
- **Orchestration** : Stratégies de sélection et terminaison des agents

### Dependances

```python
# Installation
pip install semantic-kernel openai python-dotenv
```

## 📊 État Partagé : La Classe `ProofState`

La classe `ProofState` est le **cœur du système**. Elle contient :

### 1. Phase de preuve (`ProofPhase` enum)
```
INIT → SEARCH → TACTIC_GEN → VERIFICATION → REFINEMENT → COMPLETE
```

Chaque phase détermine **quel agent agit** :
- `INIT` → CoordinatorAgent décide de la stratégie
- `SEARCH` → SearchAgent cherche des lemmes
- `TACTIC_GEN` → TacticAgent génère une tactique
- `VERIFICATION` → VerifierAgent teste la preuve
- `REFINEMENT` → CriticAgent analyse et ajuste
- `COMPLETE` → Session terminée

### 2. Stratégie de preuve (`ProofStrategy` enum)

```python
EXPLORATION   # Recherche large de lemmes
REFINEMENT    # Ajustement d'une preuve existante
VALIDATION    # Vérification formelle
RECOVERY      # Récupération après erreur
```

La stratégie influence **quels lemmes rechercher** et **quelles tactiques essayer**.

### 3. Historique et métriques

- `tactic_history` : Liste de toutes les tactiques essayées (succès + échecs)
- `verification_results` : Résultats des vérifications Lean
- `current_proof` : Preuve en construction
- `error_count` : Nombre d'erreurs rencontrées

### 4. Snapshots JSON

À chaque itération, on peut sauvegarder l'état complet en JSON :

```json
{
  "phase": "TACTIC_GEN",
  "strategy": "EXPLORATION",
  "itération": 5,
  "current_goal": "n + 0 = n",
  "tactic_history": [...],
  "current_proof": ["intro n", "rw [Nat.add_zero]"]
}
```

**Utilité** : Debugging, reproduction de bugs, benchmarking.

In [1]:
# =============================================================================
# Section 1 - ProofState: État Partage pour Multi-Agents
# =============================================================================
# Pattern inspire de RhetoricalAnalysisState dans Argument_Analysis
# Permet la synchronisation entre agents avec designation explicite

import os
import sys
import json
import time
import uuid
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Tuple
from datetime import datetime
from enum import Enum
from pathlib import Path

# --- Detection robuste du repertoire du notebook ---
# Fonctionne sous Windows, Linux, macOS et WSL (Epic #2314, Issue #2315)
notebook_dir = None

# Strategie 1: Variable environnement LEAN_NOTEBOOK_DIR
if os.getenv("LEAN_NOTEBOOK_DIR"):
    notebook_dir = Path(os.getenv("LEAN_NOTEBOOK_DIR"))
    if not (notebook_dir / "lean_runner.py").exists():
        notebook_dir = None

# Strategie 2: Chercher dans cwd et parents (cross-platform)
if not notebook_dir:
    cwd = Path.cwd().resolve()
    current = cwd
    for _ in range(10):
        candidate = current / "MyIA.AI.Notebooks" / "SymbolicAI" / "Lean"
        if candidate.exists() and (candidate / "lean_runner.py").exists():
            notebook_dir = candidate
            break
        current = current.parent
        if current == current.parent:
            break

# Strategie 3: Chemins hardcodes (fallback dynamique, Epic #2314)
if not notebook_dir:
    _drive = Path.cwd().resolve().drive
    for _drive_letter in ([_drive[0]] if _drive else ['c', 'd']):
        for _root in [f"{_drive_letter}:/dev/CoursIA", f"{_drive_letter.upper()}/dev/CoursIA"]:
            _candidate = Path(_root) / "MyIA.AI.Notebooks" / "SymbolicAI" / "Lean"
            if _candidate.exists() and (_candidate / "lean_runner.py").exists():
                notebook_dir = _candidate
                break
        if notebook_dir:
            break

if not notebook_dir:
    notebook_dir = Path.cwd()
    print(f"[WARN] lean_runner.py non trouve, fallback sur: {notebook_dir}")

# --- Charger .env ---
try:
    from dotenv import load_dotenv
    env_paths = [
        notebook_dir / ".env",
        notebook_dir.parent / ".env",
        Path.home() / ".env"
    ]
    for p in env_paths:
        if p.exists():
            load_dotenv(p, override=True)
            print(f"Configuration chargee depuis: {p}")
            break
    else:
        print("Aucun fichier .env trouve")
except ImportError:
    print("python-dotenv non installe")

# --- Importer lean_runner.py ---
if notebook_dir and str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

try:
    from lean_runner import LeanRunner, LeanResult
    # Chemin relatif au repo (machine-independant, pas de chemin absolu dans les sorties)
    try:
        _display_dir = notebook_dir.relative_to(notebook_dir.parents[2])
    except (IndexError, ValueError):
        _display_dir = notebook_dir.name
    print(f"lean_runner importe avec succes depuis {_display_dir}")
except ImportError as e:
    print(f"ERREUR: Impossible d'importer lean_runner: {e}")
    print(f"Repertoire de travail: {Path.cwd()}")
    print(f"notebook_dir detecte: {notebook_dir}")
    print(f"sys.path: {sys.path[:5]}")
    raise

# --- Enumerations ---

class ProofStrategy(Enum):
    """Strategie de preuve en cours."""
    EXPLORATION = "exploration"      # Recherche initiale de lemmes
    REFINEMENT = "refinement"        # Affinage des tactiques
    VALIDATION = "validation"        # Vérification finale
    RECOVERY = "recovery"            # Recuperation apres echecs

class TacticDifficulty(Enum):
    """Niveau de difficulte des tactiques."""
    SIMPLE = "simple"      # rfl, exact, omega
    INTERMEDIATE = "intermediate"  # simp, ring, linarith
    ADVANCED = "advanced"  # induction, cases

class ProofPhase(Enum):
    """Phase de la boucle de preuve."""
    INIT = "init"
    SEARCH = "search"
    GENERATE = "generate"
    VERIFY = "verify"
    ANALYZE = "analyze"
    COMPLETE = "complete"
    FAILED = "failed"

# --- ProofState: État partage entre agents ---

@dataclass
class TacticAttempt:
    """Une tentative de tactique."""
    tactic: str
    success: bool
    error: Optional[str] = None
    timestamp: datetime = field(default_factory=datetime.now)
    state_before: Optional[str] = None
    confidence: Optional[float] = None
    explanation: Optional[str] = None

@dataclass
class ProofState:
    """
    État partage entre les agents pour la preuve d'un théorème.
    Permet la coordination sans couplage fort.
    """
    # Identifiants
    session_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    theorem_name: str = ""
    theorem_statement: str = ""

    # État de la preuve
    current_goal: str = ""
    current_proof: List[str] = field(default_factory=list)
    phase: ProofPhase = ProofPhase.INIT
    strategy: ProofStrategy = ProofStrategy.EXPLORATION

    # Résultats des agents
    discovered_lemmas: List[str] = field(default_factory=list)
    generated_tactics: List[str] = field(default_factory=list)
    tactic_history: List[TacticAttempt] = field(default_factory=list)

    # Metriques
    iteration: int = 0
    max_iterations: int = 10
    start_time: datetime = field(default_factory=datetime.now)

    # Erreurs et diagnostics
    last_error: Optional[str] = None
    final_proof: Optional[str] = None
    error_count: int = 0

    # Vérification tracking
    verification_results: List[Dict[str, Any]] = field(default_factory=list)
    total_lean_time_ms: float = 0.0

    # Agent designation for orchestration
    _next_agent: Optional[str] = field(default=None, repr=False)

    def add_tactic_attempt(self, tactic: str, state_before: Optional[str] = None,
                           confidence: Optional[float] = None, explanation: Optional[str] = None,
                           success: bool = False, error: Optional[str] = None) -> str:
        """Enregistre une tentative de tactique."""
        attempt_id = f"attempt_{len(self.tactic_history) + 1}"
        self.tactic_history.append(TacticAttempt(
            tactic=tactic,
            success=success,
            error=error,
            state_before=state_before,
            confidence=confidence,
            explanation=explanation
        ))
        if success:
            self.current_proof.append(tactic)
        else:
            self.error_count += 1
            self.last_error = error
        return attempt_id

    def add_lemma(self, name: str, statement: str, namespace: str = "", relevance: float = 0.5) -> str:
        """Ajoute un lemme decouvert a la liste."""
        lemma_id = f"{namespace}.{name}" if namespace else name
        lemma_info = f"{lemma_id}: {statement} (relevance: {relevance})"
        if lemma_info not in self.discovered_lemmas:
            self.discovered_lemmas.append(lemma_info)
        return lemma_id

    def get_context_summary(self) -> str:
        """Resume le contexte pour les agents."""
        return f"""
Théorème: {self.theorem_name}
Enonce: {self.theorem_statement}
But actuel: {self.current_goal}
Phase: {self.phase.value}
Strategie: {self.strategy.value}
Iteration: {self.iteration}/{self.max_iterations}
Tactiques reussies: {len(self.current_proof)}
Erreurs: {self.error_count}
Derniere erreur: {self.last_error or 'Aucune'}
""".strip()


    

    # --- Properties for compatibility ---
    @property
    def tactics_history(self) -> List[TacticAttempt]:
        """Alias pour tactic_history (compatibilite)."""
        return self.tactic_history

    @property
    def proof_complete(self) -> bool:
        """True si la preuve est complète."""
        return self.phase == ProofPhase.COMPLETE
    
    @proof_complete.setter
    def proof_complete(self, value: bool):
        """Définit la completion de la preuve."""
        if value:
            self.phase = ProofPhase.COMPLETE
        elif self.phase == ProofPhase.COMPLETE:
            self.phase = ProofPhase.VERIFY
    
    @property
    def iteration_count(self) -> int:
        """Alias pour iteration (compatibilite)."""
        return self.iteration
    
    @iteration_count.setter
    def iteration_count(self, value: int):
        """Définit le compteur d'iterations."""
        self.iteration = value

    def increment_iteration(self):
        """Incremente le compteur d'iterations."""
        self.iteration += 1
    
    def designate_next_agent(self, agent_name: str):
        """Designe l'agent qui doit intervenir ensuite."""
        self._next_agent = agent_name
    
    def consume_next_agent_designation(self) -> Optional[str]:
        """Retourne et efface la designation d'agent."""
        agent = self._next_agent
        self._next_agent = None
        return agent
    
    def get_state_snapshot(self, summarize: bool = True) -> Dict[str, Any]:
        """Retourne un snapshot de l'état pour les plugins."""
        if summarize:
            return {
                "session_id": self.session_id,
                "theorem": self.theorem_statement,
                "goal": self.current_goal,
                "phase": self.phase.value,
                "strategy": self.strategy.value,
                "iteration": f"{self.iteration}/{self.max_iterations}",
                "proof_steps": len(self.current_proof),
                "discovered_lemmas": len(self.discovered_lemmas),
                "errors": self.error_count,
                "last_error": self.last_error
            }
        else:
            return self.to_dict()


    def add_verification(self, attempt_id: str, success: bool, output: str, errors: str,
                         remaining_goals: Optional[str] = None, exec_time_ms: float = 0.0,
                         mode: str = "subprocess") -> str:
        """Enregistre un résultat de vérification Lean."""
        verif_id = f"verif_{len(self.verification_results) + 1}"
        self.verification_results.append({
            "id": verif_id,
            "attempt_id": attempt_id,
            "success": success,
            "output": output,
            "errors": errors,
            "remaining_goals": remaining_goals,
            "exec_time_ms": exec_time_ms,
            "mode": mode,
            "timestamp": datetime.now().isoformat()
        })
        return verif_id


    def set_proof_complete(self, proof: str):
        """Marque la preuve comme terminée et change la phase."""
        self.final_proof = proof
        self.phase = ProofPhase.COMPLETE


    def set_strategy(self, strategy: 'ProofStrategy'):
        """Change la stratégie de preuve."""
        self.strategy = strategy

    def to_dict(self) -> Dict[str, Any]:
        """Serialise l'état."""
        return {
            "session_id": self.session_id,
            "theorem_name": self.theorem_name,
            "theorem_statement": self.theorem_statement,
            "current_goal": self.current_goal,
            "current_proof": self.current_proof,
            "phase": self.phase.value,
            "strategy": self.strategy.value,
            "discovered_lemmas": self.discovered_lemmas,
            "generated_tactics": self.generated_tactics,
            "iteration": self.iteration,
            "max_iterations": self.max_iterations,
            "error_count": self.error_count,
            "last_error": self.last_error
        }

# --- Test de l'initialisation ---
print("\n" + "="*60)
print("ProofState initialise avec succes")
print(f"LeanRunner disponible: {LeanRunner is not None}")
print("="*60)


Configuration chargee depuis: D:\dev\CoursIA-17357-lean\MyIA.AI.Notebooks\SymbolicAI\Lean\.env


lean_runner importe avec succes depuis MyIA.AI.Notebooks\SymbolicAI\Lean

ProofState initialise avec succes
LeanRunner disponible: True


### 1.1. Vue d'ensemble des Plugins

L'architecture utilise 4 plugins specialises, chacun exposant des fonctions via `@kernel_function`:

| Plugin | Rôle | Fonctions cles |
|--------|------|----------------|
| **ProofStateManagerPlugin** | Gestion de l'état | get_proof_state, add_lemma, designate_next_agent |
| **LeanSearchPlugin** | Recherche Mathlib | search_mathlib_lemmas, check_lemma_type |
| **LeanTacticPlugin** | Generation tactiques | generate_tactics, analyze_tactic_failure |
| **LeanVerificationPlugin** | Vérification Lean | verify_proof, verify_tactic_step |

Ce pattern permet aux agents d'appeler ces fonctions automatiquement grace au `FunctionChoiceBehavior.Auto()` de Semantic Kernel.

## 🔌 Plugins Semantic Kernel : Exposer l'État aux Agents

### Problème

Les agents LLM ne peuvent pas accéder directement à `ProofState` (objet Python).

### Solution : Plugins

Un **plugin Semantic Kernel** expose des méthodes Python comme **fonctions appelables par le LLM**.

```python
@kernel_function(
    description="Enregistre une tentative de tactique",
    name="log_tactic_attempt"
)
def log_tactic_attempt(self, tactic: str, confidence: float) -> str:
    attempt_id = self._state.add_tactic_attempt(tactic, confidence=confidence)
    return f"Tactique {tactic} enregistrée avec ID {attempt_id}"
```

### Décorateur `@kernel_function`

- `description` : Ce que le LLM voit ("À quoi sert cette fonction ?")
- `name` : Nom de la fonction pour le LLM
- Paramètres : Doivent correspondre **EXACTEMENT** à ce que le plugin appelle

### Les 4 plugins

1. **log_tactic_attempt** : Enregistrer une tactique essayée
2. **add_verification_result** : Enregistrer le résultat Lean
3. **set_proof_strategy** : Changer la stratégie de recherche
4. **mark_proof_complete** : Déclarer la preuve terminée

### Pourquoi c'est critique ?

Sans plugins, le LLM ne peut que **parler** de preuves. Avec plugins, il peut **agir** :

- Essayer des tactiques
- Vérifier formellement
- Ajuster sa stratégie en temps réel

In [2]:
# =============================================================================
# Section 1.1-1.4 - Plugins Semantic Kernel
# =============================================================================
# Architecture en 4 plugins specialises:
# - ProofStateManagerPlugin: Gestion de l'état partage
# - LeanSearchPlugin: Recherche de lemmes Mathlib
# - LeanTacticPlugin: Generation de tactiques
# - LeanVerificationPlugin: Vérification avec lean_runner.py

# Import du decorateur kernel_function
try:
    from semantic_kernel.functions import kernel_function
    SK_AVAILABLE = True
    print("Semantic Kernel disponible - utilisation des vrais decorateurs")
except ImportError:
    SK_AVAILABLE = False
    print("Semantic Kernel non disponible - mode simulation")
    # Decorateur de simulation
    def kernel_function(description="", name=None):
        def decorator(func):
            func._sk_function = True
            func._sk_description = description
            func._sk_name = name or func.__name__
            return func
        return decorator

# =============================================================================
# 8.2 ProofStateManagerPlugin
# =============================================================================

class ProofStateManagerPlugin:
    """
    Plugin pour gerer l'état partage de la preuve.
    Expose les méthodes de ProofState via @kernel_function.
    """

    def __init__(self, state: ProofState):
        self._state = state

    @kernel_function(
        description="Obtient un apercu de l'état actuel de la preuve (théorème, lemmes, tactiques, etc.)",
        name="get_proof_state"
    )
    def get_proof_state(self, summarize: bool = True) -> str:
        """Retourne l'état actuel sous forme JSON."""
        snapshot = self._state.get_state_snapshot(summarize=summarize)
        return json.dumps(snapshot, indent=2, ensure_ascii=False)

    @kernel_function(
        description="Ajoute un lemme decouvert a l'état partage",
        name="add_discovered_lemma"
    )
    def add_discovered_lemma(
        self, name: str, statement: str, namespace: str = "", relevance: float = 0.5
    ) -> str:
        """Enregistre un lemme trouve par SearchAgent."""
        lemma_id = self._state.add_lemma(name, statement, namespace, relevance)
        return f"Lemme ajoute: {lemma_id} ({name})"

    @kernel_function(
        description="Enregistre une tentative de tactique avec son niveau de confiance",
        name="log_tactic_attempt"
    )
    def log_tactic_attempt(
        self, tactic: str, state_before: str, confidence: float = 0.5, explanation: str = ""
    ) -> str:
        """Enregistre une tactique tentee par TacticAgent."""
        attempt_id = self._state.add_tactic_attempt(tactic, state_before, confidence, explanation)
        return f"Tactique enregistree: {attempt_id}"

    @kernel_function(
        description="Enregistre le résultat d'une vérification Lean",
        name="add_verification_result"
    )
    def add_verification_result(
        self, attempt_id: str, success: bool, output: str, errors: str,
        remaining_goals: str = "", exec_time_ms: float = 0.0
    ) -> str:
        """Enregistre un résultat de vérification."""
        verif_id = self._state.add_verification(
            attempt_id, success, output, errors,
            remaining_goals if remaining_goals else None, exec_time_ms, "subprocess"
        )
        status = "OK" if success else "ECHEC"
        return f"Verification {verif_id}: {status}"

    @kernel_function(
        description="Designe l'agent qui doit parler au prochain tour. IMPORTANT: utiliser le nom exact.",
        name="designate_next_agent"
    )
    def designate_next_agent(self, agent_name: str) -> str:
        """Delegue au prochain agent."""
        valid_agents = ["SearchAgent", "TacticAgent", "VerifierAgent", "CriticAgent", "CoordinatorAgent"]
        if agent_name not in valid_agents:
            return f"ERREUR: Agent invalide '{agent_name}'. Valides: {valid_agents}"
        self._state.designate_next_agent(agent_name)
        return f"Prochain agent: {agent_name}"

    @kernel_function(
        description="Marque la preuve comme terminee avec le code final",
        name="set_proof_complete"
    )
    def set_proof_complete(self, proof_code: str) -> str:
        """Marque la preuve comme reussie."""
        self._state.set_proof_complete(proof_code)
        return f"PREUVE COMPLETE! Code: {proof_code[:100]}..."

    @kernel_function(
        description="Change la strategie de preuve (exploration, refinement, validation, recovery)",
        name="set_proof_strategy"
    )
    def set_proof_strategy(self, strategy: str) -> str:
        """Change la strategie de preuve."""
        try:
            self._state.set_strategy(ProofStrategy(strategy))
            return f"Strategie changee: {strategy}"
        except ValueError:
            return f"ERREUR: Strategie invalide '{strategy}'. Valides: exploration, refinement, validation, recovery"


Semantic Kernel disponible - utilisation des vrais decorateurs


### 1.2. LeanSearchPlugin : Recherche de Lemmes Mathlib

**Plugin exposant les méthodes de recherche** pour SearchAgent.

#### Méthodes Exposées

```python
@kernel_function
def search_lemmas(goal: str, keywords: List[str]) -> List[Lemma]:
    # Recherche dans Mathlib par keywords
    # Retourne lemmes triés par pertinence
```

**Pattern** : SearchAgent appelle ce plugin pour trouver lemmes Mathlib pertinents.


In [3]:


# =============================================================================
# 8.3 LeanSearchPlugin
# =============================================================================

class LeanSearchPlugin:
    """
    Plugin pour la recherche de lemmes dans Mathlib.
    Utilise des patterns connus + vérification #check via lean_runner.
    """

    def __init__(self, runner: LeanRunner):
        self._runner = runner
        # Base de lemmes connus (extensible)
        self._known_lemmas = {
            # Arithmetique de base
            "Nat.add_zero": ("n + 0 = n", "Nat"),
            "Nat.zero_add": ("0 + n = n", "Nat"),
            "Nat.add_comm": ("n + m = m + n", "Nat"),
            "Nat.add_assoc": ("(n + m) + k = n + (m + k)", "Nat"),
            "Nat.mul_one": ("n * 1 = n", "Nat"),
            "Nat.one_mul": ("1 * n = n", "Nat"),
            "Nat.mul_comm": ("n * m = m * n", "Nat"),
            "Nat.mul_assoc": ("(n * m) * k = n * (m * k)", "Nat"),
            "Nat.left_distrib": ("n * (m + k) = n * m + n * k", "Nat"),
            "Nat.right_distrib": ("(n + m) * k = n * k + m * k", "Nat"),
            # Logique
            "And.intro": ("a -> b -> a /\\ b", "Logic"),
            "And.left": ("a /\\ b -> a", "Logic"),
            "And.right": ("a /\\ b -> b", "Logic"),
            "Or.inl": ("a -> a \\/ b", "Logic"),
            "Or.inr": ("b -> a \\/ b", "Logic"),
            "Eq.refl": ("a = a", "Logic"),
            "Eq.symm": ("a = b -> b = a", "Logic"),
            "Eq.trans": ("a = b -> b = c -> a = c", "Logic"),
        }

    @kernel_function(
        description="Recherche des lemmes Mathlib pertinents pour un but donné",
        name="search_mathlib_lemmas"
    )
    def search_mathlib_lemmas(self, goal: str, max_results: int = 10) -> str:
        """
        Recherche des lemmes par mots-cles.

        Args:
            goal: Description du but ou mots-cles (ex: "addition commutative")
            max_results: Nombre maximum de résultats

        Returns:
            JSON avec les lemmes trouves
        """
        goal_lower = goal.lower()
        results = []

        # Recherche par mots-cles
        keywords = goal_lower.replace("+", "add").replace("*", "mul").replace("=", "eq").split()

        for name, (statement, namespace) in self._known_lemmas.items():
            score = 0.0
            name_lower = name.lower()

            # Scoring par mots-cles
            for kw in keywords:
                if kw in name_lower:
                    score += 0.3
                if kw in statement.lower():
                    score += 0.2

            # Patterns specifiques
            if "comm" in goal_lower and "comm" in name_lower:
                score += 0.4
            if "assoc" in goal_lower and "assoc" in name_lower:
                score += 0.4
            if "zero" in goal_lower and "zero" in name_lower:
                score += 0.3
            if "distrib" in goal_lower and "distrib" in name_lower:
                score += 0.4

            if score > 0:
                results.append({
                    "name": name,
                    "statement": statement,
                    "namespace": namespace,
                    "relevance": min(score, 1.0)
                })

        # Trier par pertinence
        results.sort(key=lambda x: x["relevance"], reverse=True)
        return json.dumps(results[:max_results], indent=2, ensure_ascii=False)

    @kernel_function(
        description="Vérifie qu'un lemme existe et retourne son type via #check",
        name="check_lemma_type"
    )
    def check_lemma_type(self, lemma_name: str) -> str:
        """
        Vérifie l'existence d'un lemme via #check.

        Args:
            lemma_name: Nom du lemme (ex: "Nat.add_comm")

        Returns:
            JSON {exists, type, error}
        """
        code = f"#check {lemma_name}"
        result = self._runner.run(code)

        if result.success and not result.errors:
            # Extraire le type de la sortie
            return json.dumps({
                "exists": True,
                "type": result.output.strip(),
                "error": None
            })
        else:
            return json.dumps({
                "exists": False,
                "type": None,
                "error": result.errors or "Lemme non trouve"
            })


print("LeanSearchPlugin: classe initialisee")

# =============================================================================
# 8.4 LeanTacticPlugin


LeanSearchPlugin: classe initialisee


## 1.3. Plugins de Tactiques et Vérification

Les deux plugins restants gerent la **generation de tactiques** et la **vérification formelle** :

### LeanTacticPlugin

- **Responsabilite** : Generer des tactiques Lean adaptees au contexte
- **Méthodes exposees** :
  - `generate_tactic()` : Genere une tactique basee sur goal + lemmes + historique
  - `estimate_confidence()` : Estime la probabilite de succes (0.0-1.0)
- **LLM-aware** : Utilise un prompt structure pour le LLM avec exemples de tactiques Lean
- **Stratégies** : `exact`, `rw`, `apply`, `simp`, `induction`, `cases`, etc.

### LeanVerificationPlugin

- **Responsabilite** : Vérifier les preuves via compilation Lean
- **Méthodes exposees** :
  - `verify_proof()` : Compile le théorème avec tactiques et retourne succes/echec
  - `parse_lean_errors()` : Parse les messages d'erreur Lean pour feedback agents
- **Detection de completion** : Reconnait "no goals" = preuve complète
- **Gestion d'erreurs** : Extrait type d'erreur (type mismatch, tactic failed, etc.) pour CriticAgent

**Flow typique** :
```
SearchAgent trouve lemmes
   |
   v
TacticAgent genere tactique (via LeanTacticPlugin)
   |
   v
VerifierAgent compile (via LeanVerificationPlugin)
   |
   +-- Success → COMPLETE
   +-- Failure → CriticAgent analyse → retry
```


In [4]:
# =============================================================================

class LeanTacticPlugin:
    """
    Plugin pour la generation de tactiques.
    Fournit des heuristiques et analyse les echecs.
    """

    def __init__(self):
        # Tactiques par difficulte
        self._tactics = {
            "simple": ["rfl", "trivial", "exact ?_", "assumption"],
            "medium": ["simp", "omega", "decide", "constructor", "intro", "apply"],
            "complex": ["ring", "linarith", "aesop", "induction", "cases", "rcases"]
        }

        # Heuristiques par pattern de but
        self._heuristics = {
            "equality": ["rfl", "exact", "simp", "ring", "omega"],
            "forall": ["intro", "intros", "apply"],
            "exists": ["use", "exists", "exact"],
            "and": ["constructor", "exact And.intro"],
            "or": ["left", "right"],
            "implication": ["intro", "apply", "exact"],
            "nat_arithmetic": ["omega", "simp", "decide"],
            "ring_expression": ["ring", "ring_nf"]
        }

    @kernel_function(
        description="Genere des tactiques appropriees pour un but donné",
        name="generate_tactics"
    )
    def generate_tactics(self, goal: str, context: str = "", difficulty: str = "simple") -> str:
        """
        Genere des tactiques pour le but courant.

        Args:
            goal: Le but Lean a prouver
            context: Contexte additionnel (lemmes disponibles, etc.)
            difficulty: simple, medium, ou complex

        Returns:
            JSON [{tactic, confidence, explanation}]
        """
        suggestions = []
        goal_lower = goal.lower()

        # Detecter le type de but
        detected_patterns = []
        if "=" in goal:
            detected_patterns.append("equality")
        if "forall" in goal_lower or "∀" in goal:
            detected_patterns.append("forall")
        if "exists" in goal_lower or "∃" in goal:
            detected_patterns.append("exists")
        if "/\\" in goal or "∧" in goal or "And" in goal:
            detected_patterns.append("and")
        if "\\/" in goal or "∨" in goal or "Or" in goal:
            detected_patterns.append("or")
        if "->" in goal or "→" in goal:
            detected_patterns.append("implication")
        if any(x in goal_lower for x in ["nat", "n +", "m +", "+ 0", "0 +"]):
            detected_patterns.append("nat_arithmetic")
        if any(x in goal for x in ["*", "+"]) and "=" in goal:
            detected_patterns.append("ring_expression")

        # Collecter les tactiques suggeres
        seen = set()
        for pattern in detected_patterns:
            for tactic in self._heuristics.get(pattern, []):
                if tactic not in seen:
                    seen.add(tactic)
                    confidence = 0.7 if difficulty == "simple" else 0.5
                    suggestions.append({
                        "tactic": tactic,
                        "confidence": confidence,
                        "explanation": f"Pattern detecte: {pattern}"
                    })

        # Ajouter des tactiques de base
        base_tactics = self._tactics.get(difficulty, self._tactics["simple"])
        for tactic in base_tactics[:3]:
            if tactic not in seen:
                suggestions.append({
                    "tactic": tactic,
                    "confidence": 0.3,
                    "explanation": f"Tactique {difficulty} générique"
                })

        # Trier par confiance
        suggestions.sort(key=lambda x: x["confidence"], reverse=True)
        return json.dumps(suggestions[:8], indent=2, ensure_ascii=False)

    @kernel_function(
        description="Analyse un echec de tactique et suggere des alternatives",
        name="analyze_tactic_failure"
    )
    def analyze_tactic_failure(self, failed_tactic: str, error_msg: str) -> str:
        """
        Analyse pourquoi une tactique a echoue.

        Args:
            failed_tactic: La tactique qui a echoue
            error_msg: Message d'erreur Lean

        Returns:
            JSON {diagnosis, alternatives, error_type}
        """
        error_lower = error_msg.lower()
        diagnosis = ""
        alternatives = []
        error_type = "unknown"

        # Classifier l'erreur
        if "unknown identifier" in error_lower or "unknown constant" in error_lower:
            error_type = "unknown_identifier"
            diagnosis = "Lemme ou identifiant non reconnu. Vérifier l'import ou le nom."
            alternatives = ["Chercher le bon nom avec #check", "Vérifier les imports"]

        elif "type mismatch" in error_lower:
            error_type = "type_mismatch"
            diagnosis = "Les types ne correspondent pas. Vérifier les arguments."
            alternatives = ["exact", "apply", "simp"]

        elif "unsolved goals" in error_lower or "goals remain" in error_lower:
            error_type = "unsolved_goals"
            diagnosis = "Des sous-buts restent. La tactique n'a pas complété la preuve."
            alternatives = ["Ajouter d'autres tactiques", "Essayer simp", "Decomposer avec have"]

        elif "tactic failed" in error_lower:
            error_type = "tactic_failed"
            diagnosis = f"La tactique '{failed_tactic}' n'a pas pu s'appliquer."
            # Suggerer des alternatives
            if failed_tactic in ["ring", "linarith"]:
                alternatives = ["omega", "simp", "decide"]
            elif failed_tactic == "simp":
                alternatives = ["simp only", "rfl", "exact"]
            else:
                alternatives = ["simp", "omega", "exact ?_"]

        elif "declaration uses 'sorry'" in error_lower:
            error_type = "sorry"
            diagnosis = "La preuve contient 'sorry' - incomplète."
            alternatives = ["Compléter la preuve", "Remplacer sorry par une vraie tactique"]

        else:
            error_type = "other"
            diagnosis = f"Erreur non classifiee: {error_msg[:100]}"
            alternatives = ["Vérifier la syntaxe", "Essayer une approche différente"]

        return json.dumps({
            "diagnosis": diagnosis,
            "alternatives": alternatives,
            "error_type": error_type,
            "original_error": error_msg[:200]
        }, indent=2, ensure_ascii=False)

print("LeanTacticPlugin: classe initialisee")


LeanTacticPlugin: classe initialisee


### 1.4. LeanVerificationPlugin : Compilation et Vérification

**Plugin exposant les méthodes de vérification** pour VerifierAgent.

#### Méthodes Exposées

```python
@kernel_function
def verify_proof(theorem: str, tactics: str) -> VerificationResult:
    # Compile le théorème avec tactiques
    # Parse output Lean (success/errors)
    # Détecte "no goals" = preuve complète
```

**Pattern** : VerifierAgent appelle ce plugin pour compiler preuves avec LeanRunner.


In [5]:


# =============================================================================
# 8.5 LeanVerificationPlugin
# =============================================================================

class LeanVerificationPlugin:
    """
    Plugin pour la vérification des preuves avec lean_runner.
    """

    def __init__(self, runner: LeanRunner):
        self._runner = runner

    @kernel_function(
        description="Vérifie une preuve complète (théorème + tactiques)",
        name="verify_proof"
    )
    def verify_proof(self, theorem_statement: str, proof_tactics: str) -> str:
        """
        Vérifie un théorème avec sa preuve.

        Args:
            theorem_statement: L'enonce du théorème (ex: "theorem add_zero (n : Nat) : n + 0 = n")
            proof_tactics: La preuve (ex: "exact Nat.add_zero n")

        Returns:
            JSON {success, output, errors, exec_time_ms, backend}
        """
        import time

        # Construire le code complet
        if "by" not in proof_tactics and ":=" not in proof_tactics:
            code = f"{theorem_statement} := by {proof_tactics}"
        elif ":=" in proof_tactics:
            code = f"{theorem_statement} {proof_tactics}"
        else:
            code = f"{theorem_statement} := {proof_tactics}"

        start = time.time()
        result = self._runner.run(code)
        exec_time = (time.time() - start) * 1000

        return json.dumps({
            "success": result.success,
            "output": result.output,
            "errors": result.errors,
            "exit_code": result.exit_code,
            "exec_time_ms": round(exec_time, 2),
            "backend": result.backend,
            "code": code
        }, indent=2, ensure_ascii=False)

    @kernel_function(
        description="Vérifie une etape de tactique incrementale",
        name="verify_tactic_step"
    )
    def verify_tactic_step(
        self, partial_proof: str, next_tactic: str, theorem_statement: str
    ) -> str:
        """
        Vérifie une tactique incrementale.

        Args:
            partial_proof: Les tactiques deja appliquees (separees par ;)
            next_tactic: La prochaine tactique a essayer
            theorem_statement: L'enonce du théorème

        Returns:
            JSON {tactic_valid, remaining_goals, error, exec_time_ms}
        """
        import time

        # Combiner les tactiques
        if partial_proof:
            all_tactics = f"{partial_proof}; {next_tactic}"
        else:
            all_tactics = next_tactic

        code = f"{theorem_statement} := by {all_tactics}"

        start = time.time()
        result = self._runner.run(code)
        exec_time = (time.time() - start) * 1000

        # Analyser les goals restants
        remaining_goals = None
        if "unsolved goals" in result.errors.lower():
            # Extraire les goals du message d'erreur
            remaining_goals = result.errors

        return json.dumps({
            "tactic_valid": result.success or "unsolved goals" not in result.errors.lower(),
            "remaining_goals": remaining_goals,
            "error": result.errors if not result.success else None,
            "exec_time_ms": round(exec_time, 2),
            "applied_tactics": all_tactics
        }, indent=2, ensure_ascii=False)


# =============================================================================
# Test des Plugins
# =============================================================================

print("\n=== Test des Plugins ===")

# Creer l'état et le runner
test_state = ProofState(theorem_statement="theorem test_add (n : Nat) : n + 0 = n")
runner = LeanRunner(backend="subprocess", timeout=30)

# Instancier les plugins
state_plugin = ProofStateManagerPlugin(test_state)
search_plugin = LeanSearchPlugin(runner)
tactic_plugin = LeanTacticPlugin()
verif_plugin = LeanVerificationPlugin(runner)

# Test 1: Recherche de lemmes
print("\n1. Recherche de lemmes pour 'addition zero':")
lemmas = search_plugin.search_mathlib_lemmas("addition zero", max_results=3)
print(lemmas)

# Test 2: Generation de tactiques
print("\n2. Tactiques pour 'n + 0 = n':")
tactics = tactic_plugin.generate_tactics("n + 0 = n", difficulty="simple")
print(tactics)

# Test 3: Vérification avec lean_runner
print("\n3. Verification d'une preuve:")
result = verif_plugin.verify_proof("theorem test_rfl : 2 + 2 = 4", "rfl")
print(result)

# Test 4: Plugin StateManager
print("\n4. Ajout via StateManagerPlugin:")
print(state_plugin.add_discovered_lemma("Nat.add_zero", "n + 0 = n", "Nat", 0.9))
print(state_plugin.get_proof_state(summarize=True))



=== Test des Plugins ===



1. Recherche de lemmes pour 'addition zero':
[
  {
    "name": "Nat.add_zero",
    "statement": "n + 0 = n",
    "namespace": "Nat",
    "relevance": 0.6
  },
  {
    "name": "Nat.zero_add",
    "statement": "0 + n = n",
    "namespace": "Nat",
    "relevance": 0.6
  }
]

2. Tactiques pour 'n + 0 = n':
[
  {
    "tactic": "rfl",
    "confidence": 0.7,
    "explanation": "Pattern detecte: equality"
  },
  {
    "tactic": "exact",
    "confidence": 0.7,
    "explanation": "Pattern detecte: equality"
  },
  {
    "tactic": "simp",
    "confidence": 0.7,
    "explanation": "Pattern detecte: equality"
  },
  {
    "tactic": "ring",
    "confidence": 0.7,
    "explanation": "Pattern detecte: equality"
  },
  {
    "tactic": "omega",
    "confidence": 0.7,
    "explanation": "Pattern detecte: equality"
  },
  {
    "tactic": "decide",
    "confidence": 0.7,
    "explanation": "Pattern detecte: nat_arithmetic"
  },
  {
    "tactic": "ring_nf",
    "confidence": 0.7,
    "explanation": "Patter

{
  "success": true,
  "output": "",
  "errors": "",
  "exit_code": 0,
  "exec_time_ms": 6157.29,
  "backend": "subprocess",
  "code": "theorem test_rfl : 2 + 2 = 4 := by rfl"
}

4. Ajout via StateManagerPlugin:
Lemme ajoute: Nat.Nat.add_zero (Nat.add_zero)
{
  "session_id": "15458973",
  "theorem": "theorem test_add (n : Nat) : n + 0 = n",
  "goal": "",
  "phase": "init",
  "strategy": "exploration",
  "iteration": "0/10",
  "proof_steps": 0,
  "discovered_lemmas": 1,
  "errors": 0,
  "last_error": null
}


## Exercice 1 : Créer un plugin de classement de lemmes

En vous basant sur les plugins `LeanSearchPlugin` et `ProofStateManagerPlugin` presentes dans la Section 1,
créez un **`LemmaRankerPlugin`** qui classe les lemmes trouves par relevance.

**Objectif** : Implementer une fonction `rank_lemmas` qui prend une liste de lemmes candidats
et retourne les lemmes tries par pertinence (nombre de mots-cles en commun avec l'enonce).

**Étapes** :
1. Définir la fonction `rank_lemmas(lemmas: list[str], theorem_statement: str) -> list[str]`
2. Extraire les mots-cles de `theorem_statement` (mots > 3 lettres)
3. Calculer un score de pertinence pour chaque lemme (nombre de mots-cles partages)
4. Retourner les lemmes tries par score descendant

**Indice** : Utilisez `set()` pour l'intersection de mots-cles et `sorted(..., key=...)` avec `reverse=True`.


In [6]:
# ============================================================
# Exercice 1 : LemmaRankerPlugin
# ============================================================
# Classez les lemmes par pertinence relative a l enonce.
# ============================================================

def rank_lemmas(lemmas: list, theorem_statement: str) -> list:
    """
    Classe les lemmes par pertinence (nombre de mots-cles
    en commun avec l enonce de la théorème).

    Args:
        lemmas: Liste des noms de lemmes candidats
        theorem_statement: Enonce du théorème a prouver

    Returns:
        Liste des lemmes tries par pertinence decroissante
    """
    # TODO: Etape 1 - Extraire les mots-cles (> 3 lettres) de theorem_statement
    keywords = set()  # TODO étudiant

    # TODO: Etape 2 - Calculer le score de chaque lemme
    scores = {}  # TODO étudiant

    # TODO: Etape 3 - Retourner les lemmes tries par score descendant
    return sorted(lemmas, key=lambda l: 0, reverse=True)  # TODO etudiant


# Test rapide
test_lemmas = ["mul_comm", "add_comm", "mul_assoc", "add_assoc", "mul_zero"]
test_stmt = "forall a b : Nat, a * b = b * a"
result = rank_lemmas(test_lemmas, test_stmt)
print(f"Lemmes classes : {result}")


Lemmes classes : ['mul_comm', 'add_comm', 'mul_assoc', 'add_assoc', 'mul_zero']


## 2. Architecture : 5 Agents Specialises

Le système multi-agents comprend 5 rôles distincts:

| Agent | Rôle | Plugins | Delegation |
|-------|------|---------|------------|
| **SearchAgent** | Recherche lemmes Mathlib | LeanSearch, StateManager | TacticAgent si lemmes trouves |
| **TacticAgent** | Generation tactiques | LeanTactic, StateManager | VerifierAgent pour validation |
| **VerifierAgent** | Vérification Lean | LeanVerification, StateManager | CriticAgent si echec |
| **CriticAgent** | Analyse echecs | LeanTactic, StateManager | Redirection selon erreur |
| **CoordinatorAgent** | Supervision globale | StateManager | Gestion des blocages |

**Pattern cle**: Chaque agent designe explicitement le suivant via `designate_next_agent()`.

## 🤖 Création des Agents Semantic Kernel

### Anatomie d'un agent

Chaque agent a :

1. **Un nom** : "SearchAgent", "TacticAgent", etc.
2. **Des instructions** : Prompt système qui définit son rôle
3. **Des plugins** : Fonctions qu'il peut appeler (via StatePlugin)
4. **Un modèle LLM** : GPT-5.2, Claude, etc.

### Exemple : SearchAgent

```python
search_agent = kernel.add_agent(
    name="SearchAgent",
    instructions="""Tu es un expert en recherche de lemmes Mathlib.
    Ton rôle : Trouver les lemmes pertinents pour le but actuel.
    Délègue à TacticAgent une fois les lemmes trouvés.""",
    plugins=[state_plugin]
)
```

### Instructions : Le "métier" de l'agent

Les instructions définissent :

- **Responsabilité** : "Recherche de lemmes" vs "Génération de tactiques"
- **Critères de succès** : "Trouver au moins 2 lemmes pertinents"
- **Délégation** : "Quand déléguer à un autre agent ?"

**Principe clé** : Instructions précises → Comportement prévisible

### Pattern : Stratégies basées sur l'état

Au lieu de coder en dur "SearchAgent → TacticAgent", on utilise :

```python
def select_next_agent(state: ProofState) -> str:
    if state.phase == ProofPhase.SEARCH:
        return "SearchAgent"
    elif state.phase == ProofPhase.TACTIC_GEN:
        return "TacticAgent"
    # ...
```

**Avantage** : Orchestration dynamique basée sur l'état réel de la preuve.

In [7]:
# =============================================================================
# Section 2 - Définition des 5 Agents Specialises avec Semantic Kernel
# =============================================================================
# Utilise ChatCompletionAgent de Semantic Kernel avec FunctionChoiceBehavior.Auto()

import os
import asyncio
from typing import Dict, Any, Optional

# --- Instructions des Agents ---

SEARCH_AGENT_INSTRUCTIONS = """
Tu es l'agent de RECHERCHE de lemmes pour le theorem proving en Lean 4.

TON ROLE UNIQUE:
- Chercher des lemmes Mathlib pertinents pour le théorème courant
- Identifier les lemmes qui peuvent aider a la preuve
- Enregistrer les lemmes trouves dans l'état partage

WORKFLOW:
1. Lis l'état avec get_proof_state() pour comprendre le théorème
2. Utilise search_mathlib_lemmas() avec des mots-cles pertinents
3. Vérifie les lemmes prometteurs avec check_lemma_type()
4. Enregistre les lemmes utiles avec add_discovered_lemma()
5. Delegue a TacticAgent quand tu as trouve des lemmes

IMPORTANT:
- Cherche des lemmes LIES au but (égalités, arithmetique, logique)
- Delegation: Apres avoir trouve au moins 2-3 lemmes, delegue a TacticAgent
- Si aucun lemme pertinent, delegue quand meme a TacticAgent
"""

TACTIC_AGENT_INSTRUCTIONS = """
Tu es l'agent de GENERATION DE TACTIQUES pour le theorem proving en Lean 4.

TON ROLE UNIQUE:
- Generer des sequences de tactiques Lean pour prouver le but
- Explorer les tactiques systématiquement

STRATEGIE D'EXPLORATION OBLIGATOIRE:
Tu DOIS essayer les tactiques dans cet ORDRE EXACT, meme si tu penses qu'elles echoueront:

PREMIERE TENTATIVE: Toujours essayer rfl ou trivial
DEUXIEME TENTATIVE: simp sans arguments
TROISIEME TENTATIVE: Lemmes de SearchAgent (exact Lemma_name)
QUATRIEME TENTATIVE: Tactiques avancees (omega, ring, linarith)
CINQUIEME+ TENTATIVE: Approches structurelles (induction, cases)

POURQUOI CETTE APPROCHE:
Cette strategie pedagogique montre le processus de decouverte.
Ne pas proposer la solution optimale immediatement.
Laisser le système iterer vers la solution.

WORKFLOW:
1. get_proof_state() pour comprendre le contexte
2. Choisir UNE tactique selon l'ordre ci-dessus
3. log_tactic_attempt() pour enregistrer
4. Deleguer a VerifierAgent

IMPORTANT:
- Proposer UNE SEULE tactique a la fois
- Si echec, CriticAgent analysera et guidera
- Ne pas "tricher" en donnant la reponse finale directement

"""

VERIFIER_AGENT_INSTRUCTIONS = """
Tu es l'agent de VERIFICATION pour le theorem proving en Lean 4.

TON ROLE UNIQUE:
- Vérifier les tactiques proposees avec le compilateur Lean
- Enregistrer les résultats de vérification
- Determiner si la preuve est complète ou s'il faut continuer

WORKFLOW:
1. Lis l'état avec get_proof_state() pour voir la derniere tactique
2. Utilise verify_proof() pour tester la preuve
3. Enregistre le résultat avec add_verification_result()
4. Si succes: set_proof_complete() et termine
5. Si echec: delegue a CriticAgent pour analyse

IMPORTANT:
- Teste TOUJOURS la derniere tactique proposee
- Si la preuve compile sans erreur, utilise set_proof_complete()
- Si echec, enregistre l'erreur et delegue a CriticAgent
"""

CRITIC_AGENT_INSTRUCTIONS = """
Tu es l'agent CRITIQUE pour le theorem proving en Lean 4.

TON ROLE UNIQUE:
- Analyser les echecs de vérification
- Diagnostiquer les erreurs Lean
- Orienter vers la bonne strategie de correction

WORKFLOW:
1. Lis l'état avec get_proof_state() pour voir les echecs recents
2. Utilise analyze_tactic_failure() pour comprendre l'erreur
3. Décide quelle direction prendre:
   - "unknown identifier" -> delegue a SearchAgent
   - "type mismatch" ou "tactic failed" -> delegue a TacticAgent
   - Echecs repetes (>3) -> delegue a CoordinatorAgent

IMPORTANT:
- Analyse les 3 derniers echecs pour detecter des patterns
- Si >3 echecs similaires, delegue a CoordinatorAgent
"""

COORDINATOR_AGENT_INSTRUCTIONS = """
Tu es l'agent COORDINATEUR (superviseur) pour le theorem proving en Lean 4.

TON ROLE UNIQUE:
- Superviser l'ensemble de la session de preuve
- Debloquer les situations cycliques
- Ajuster la strategie globale

QUAND TU INTERVIENS:
- Appele par CriticAgent apres echecs repetes
- Appele si max_iterations approche
- Appele pour decisions strategiques majeures

IMPORTANT:
- Tu es le dernier recours, prends des decisions audacieuses
- Si >40 iterations, suggere de simplifier le théorème
"""

# =============================================================================
# Detection de Semantic Kernel
# =============================================================================

SK_AVAILABLE = False
ANTHROPIC_AVAILABLE = False
try:
    from semantic_kernel import Kernel
    from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
    from semantic_kernel.agents.strategies import (
        KernelFunctionSelectionStrategy,
        KernelFunctionTerminationStrategy,
    )
    from semantic_kernel.agents.strategies.selection.sequential_selection_strategy import SequentialSelectionStrategy
    from semantic_kernel.agents.strategies.termination.default_termination_strategy import DefaultTerminationStrategy
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
    from semantic_kernel.connectors.ai import FunctionChoiceBehavior
    from semantic_kernel.functions import KernelFunctionFromPrompt, KernelArguments
    from semantic_kernel.contents import ChatHistoryTruncationReducer
    from semantic_kernel.agents.strategies.selection.selection_strategy import SelectionStrategy
    from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
    from semantic_kernel.contents.chat_message_content import ChatMessageContent
    from pydantic import PrivateAttr
    SK_AVAILABLE = True
    print("Semantic Kernel disponible - utilisation de ChatCompletionAgent")
    
    # Import Anthropic connector (optional)
    try:
        from semantic_kernel.connectors.ai.anthropic import AnthropicChatCompletion
        ANTHROPIC_AVAILABLE = True
        print("Anthropic connector disponible")
    except ImportError:
        print("Anthropic connector non disponible (pip install semantic-kernel[anthropic])")
        
except ImportError as e:
    print(f"Semantic Kernel non disponible: {e}")
    print("Installation: pip install semantic-kernel")


Semantic Kernel disponible - utilisation de ChatCompletionAgent
Anthropic connector disponible


## 2.1. SimpleAgent : Agent Fallback : Agent Fallback (Simulation)

**Classe de secours** pour simuler agents quand Semantic Kernel non disponible.

### Architecture

```python
class SimpleAgent:
    def __init__(self, name: str, instructions: str):
        self.name = name
        self.instructions = instructions
    
    def invoke(self, message: str) -> str:
        # Simulation simple basée sur règles
        # Utilisé en mode fallback si OpenAI API indisponible
```

**Usage** : Mode simulation pour tests sans LLM.


In [8]:

# =============================================================================
# Mode Simulation (fallback si SK non disponible)
# =============================================================================

# Flag global pour le mode démo (réponses hardcodées pédagogiques)
USE_DEMO_MODE = os.getenv("USE_DEMO_MODE", "false").lower() == "true"

class SimpleAgent:
    """
    Agent simplifie pour simulation ou fallback.

    Modes disponibles:
    - DEMO mode (USE_DEMO_MODE=True): Réponses hardcodées pour les 4 démos pédagogiques
    - Simulation mode (use_simulation=True): Logique générique basée sur les théorèmes
    - LLM mode (use_simulation=False): Appels réels à OpenAI avec function calling
    """

    def __init__(
        self,
        name: str,
        instructions: str,
        plugins: Dict[str, Any],
        use_simulation: bool = True
    ):
        self.name = name
        self.instructions = instructions
        self.plugins = plugins
        self.use_simulation = use_simulation
        self._openai_client = None

        # Initialiser le client OpenAI si mode réel
        if not use_simulation:
            try:
                from openai import OpenAI
                api_key = os.getenv("OPENAI_API_KEY")
                if api_key and len(api_key) > 10 and not api_key.startswith("sk-..."):
                    self._openai_client = OpenAI(api_key=api_key)
            except ImportError:
                pass

    def _build_openai_tools(self) -> list:
        """Construit les outils au format OpenAI function calling."""
        import inspect
        tools = []
        for plugin_name, plugin in self.plugins.items():
            for attr_name in dir(plugin):
                attr = getattr(plugin, attr_name)
                if not callable(attr):
                    continue
                # Supporter les deux décorateurs
                is_sk_func = hasattr(attr, '_sk_function') or hasattr(attr, '__kernel_function__')
                if not is_sk_func:
                    continue

                sig = inspect.signature(attr)
                properties = {}
                required = []
                for param_name, param in sig.parameters.items():
                    if param_name == 'self':
                        continue
                    param_type = "string"
                    if param.annotation != inspect.Parameter.empty:
                        if param.annotation == bool:
                            param_type = "boolean"
                        elif param.annotation in (int, float):
                            param_type = "number"
                    properties[param_name] = {
                        "type": param_type,
                        "description": f"Parameter {param_name}"
                    }
                    if param.default == inspect.Parameter.empty:
                        required.append(param_name)

                # Obtenir nom et description
                if hasattr(attr, '__kernel_function_name__'):
                    func_name = attr.__kernel_function_name__
                    func_desc = getattr(attr, "__kernel_function_description__", "")
                elif hasattr(attr, '_sk_name'):
                    func_name = attr._sk_name
                    func_desc = getattr(attr, "_sk_description", "")
                else:
                    func_name = attr_name
                    func_desc = ""

                tools.append({
                    "type": "function",
                    "function": {
                        "name": f"{plugin_name}__{func_name}",
                        "description": func_desc,
                        "parameters": {
                            "type": "object",
                            "properties": properties,
                            "required": required
                        }
                    }
                })
        return tools

    def _execute_tool_call(self, tool_name: str, arguments: dict) -> str:
        """Exécute un appel de fonction sur un plugin."""
        parts = tool_name.split("__", 1)
        if len(parts) != 2:
            return f"Erreur: format invalide: {tool_name}"

        plugin_name, func_name = parts
        plugin = self.plugins.get(plugin_name)
        if not plugin:
            return f"Erreur: plugin {plugin_name} non trouve"

        for attr_name in dir(plugin):
            attr = getattr(plugin, attr_name)
            if not callable(attr):
                continue
            is_sk = hasattr(attr, '_sk_function') or hasattr(attr, '__kernel_function__')
            if not is_sk:
                continue

            if hasattr(attr, '__kernel_function_name__'):
                name = attr.__kernel_function_name__
            elif hasattr(attr, '_sk_name'):
                name = attr._sk_name
            else:
                name = attr_name

            if name == func_name:
                try:
                    result = attr(**arguments)
                    return str(result)
                except Exception as e:
                    return f"Erreur {func_name}: {e}"

        return f"Erreur: {func_name} non trouve dans {plugin_name}"

    def invoke(self, message: str, state: ProofState) -> str:
        """Exécute l'agent sur un message."""
        state.increment_iteration()

        if self.use_simulation or not self._openai_client:
            return self._simulate_response(message, state)
        else:
            return self._call_llm(message, state)

    def _simulate_response(self, message: str, state: ProofState) -> str:
        """
        Simulation realiste basee sur l'analyse du théorème.
        Si USE_DEMO_MODE=True, utilise les réponses hardcodées pour les DEMOs.
        Sinon, utilise la logique générique.
        """
        theorem = state.theorem_statement.lower()
        goal = state.current_goal or ""

        if self.name == "SearchAgent":
            return self._do_search(state, theorem, goal)
        elif self.name == "TacticAgent":
            return self._do_tactic(state, theorem, goal)
        elif self.name == "VerifierAgent":
            return self._do_verify(state, theorem, goal)
        elif self.name == "CriticAgent":
            return self._do_critic(state, theorem)
        elif self.name == "CoordinatorAgent":
            return self._do_coordinate(state, theorem)

        return f"[{self.name}] Action simulee."

    # =========================================================================
    # SEARCH AGENT
    # =========================================================================
    def _do_search(self, state: ProofState, theorem: str, goal: str) -> str:
        """Recherche de lemmes - Mode DEMO ou générique."""
        state_mgr = self.plugins.get("state")
        search = self.plugins.get("search")

        if not state_mgr:
            return "[SearchAgent] Plugins manquants."

        # --- MODE DEMO: Réponses hardcodées pour les 4 démos pédagogiques ---
        if USE_DEMO_MODE:
            # DEMO_1: n = n (reflexivité)
            if "n = n" in theorem and "demo_rfl" in theorem:
                state_mgr.add_lemma("Eq.refl")
                state_mgr.designate_next_agent("TacticAgent")
                return "[SearchAgent] Lemme trouve: Eq.refl (reflexivite). -> TacticAgent"

            # DEMO_2: 0 + n = n (zero_add)
            if "0 + n = n" in theorem or "zero_add" in theorem:
                state_mgr.add_lemma("Nat.zero_add")
                state_mgr.add_lemma("Nat.add_zero")
                state_mgr.designate_next_agent("TacticAgent")
                return "[SearchAgent] Lemmes: Nat.zero_add, Nat.add_zero. -> TacticAgent"

            # DEMO_3: a * c + b * c = (a + b) * c (distributivité)
            if "a * c + b * c" in theorem or "add_mul_distrib" in theorem:
                state_mgr.add_lemma("Nat.add_mul")
                state_mgr.add_lemma("Nat.mul_add")
                state_mgr.add_lemma("Nat.right_distrib")
                state_mgr.designate_next_agent("TacticAgent")
                return "[SearchAgent] Lemmes distributivite: Nat.add_mul, Nat.mul_add, Nat.right_distrib. -> TacticAgent"

            # DEMO_4: m * n = n * m (commutativité multiplication)
            if "m * n = n * m" in theorem or "mul_comm_manual" in theorem:
                state_mgr.add_lemma("Nat.mul_comm")
                state_mgr.add_lemma("Nat.mul_succ")
                state_mgr.add_lemma("Nat.succ_mul")
                state_mgr.add_lemma("Nat.mul_zero")
                state_mgr.add_lemma("Nat.zero_mul")
                state_mgr.designate_next_agent("TacticAgent")
                return "[SearchAgent] Lemmes pour induction: mul_succ, succ_mul, mul_zero, zero_mul. -> TacticAgent"

        # --- MODE GÉNÉRIQUE: Logique basée sur l'analyse du théorème ---
        lemmas_found = []

        # Réflexivité
        if "n = n" in theorem or goal.strip() == "n = n":
            if hasattr(state_mgr, 'add_discovered_lemma'):
                state_mgr.add_discovered_lemma("Eq.refl", "a = a", "Logic", 1.0)
            else:
                state_mgr.add_lemma("Eq.refl")
            lemmas_found.append("Eq.refl")

        # Addition avec zéro
        if "+ 0" in theorem or "0 +" in theorem:
            if hasattr(state_mgr, 'add_discovered_lemma'):
                state_mgr.add_discovered_lemma("Nat.add_zero", "n + 0 = n", "Nat", 0.9)
                state_mgr.add_discovered_lemma("Nat.zero_add", "0 + n = n", "Nat", 0.9)
            else:
                state_mgr.add_lemma("Nat.add_zero")
                state_mgr.add_lemma("Nat.zero_add")
            lemmas_found.extend(["Nat.add_zero", "Nat.zero_add"])

        # Commutativité addition
        if "+" in theorem and ("m + n" in theorem or "n + m" in theorem or "b + a" in theorem):
            if hasattr(state_mgr, 'add_discovered_lemma'):
                state_mgr.add_discovered_lemma("Nat.add_comm", "n + m = m + n", "Nat", 0.85)
            else:
                state_mgr.add_lemma("Nat.add_comm")
            lemmas_found.append("Nat.add_comm")

        # Associativité
        if theorem.count("+") >= 2:
            if hasattr(state_mgr, 'add_discovered_lemma'):
                state_mgr.add_discovered_lemma("Nat.add_assoc", "(n + m) + k = n + (m + k)", "Nat", 0.8)
            else:
                state_mgr.add_lemma("Nat.add_assoc")
            lemmas_found.append("Nat.add_assoc")

        # Distributivité
        if "*" in theorem and "+" in theorem:
            if hasattr(state_mgr, 'add_discovered_lemma'):
                state_mgr.add_discovered_lemma("Nat.right_distrib", "(n + m) * k = n * k + m * k", "Nat", 0.9)
                state_mgr.add_discovered_lemma("Nat.add_mul", "a * c + b * c = (a + b) * c", "Nat", 0.9)
            else:
                state_mgr.add_lemma("Nat.right_distrib")
                state_mgr.add_lemma("Nat.add_mul")
            lemmas_found.extend(["Nat.right_distrib", "Nat.add_mul"])

        # Commutativité multiplication
        if "*" in theorem and ("m * n" in theorem or "n * m" in theorem):
            if hasattr(state_mgr, 'add_discovered_lemma'):
                state_mgr.add_discovered_lemma("Nat.mul_comm", "m * n = n * m", "Nat", 0.85)
            else:
                state_mgr.add_lemma("Nat.mul_comm")
            lemmas_found.append("Nat.mul_comm")

        state_mgr.designate_next_agent("TacticAgent")
        if lemmas_found:
            return f"[SearchAgent] Lemmes: {', '.join(lemmas_found[:3])}. -> TacticAgent"
        return "[SearchAgent] Recherche generique. -> TacticAgent"

    # =========================================================================
    # TACTIC AGENT
    # =========================================================================
    def _do_tactic(self, state: ProofState, theorem: str, goal: str) -> str:
        """Generation de tactiques - Mode DEMO ou générique."""
        state_mgr = self.plugins.get("state")
        if not state_mgr:
            return "[TacticAgent] Plugin manquant."

        n = len(state.tactics_history)

        # --- MODE DEMO: Séquences hardcodées pour progression pédagogique ---
        if USE_DEMO_MODE:
            # DEMO_1: n = n (reflexivity) - SUCCESS IMMEDIAT
            if "n = n" in theorem and "demo_rfl" in theorem:
                state_mgr.log_tactic_attempt("rfl", goal, 1.0, "Reflexivite directe")
                state_mgr.designate_next_agent("VerifierAgent")
                return "[TacticAgent] Tactique: rfl (reflexivite). -> VerifierAgent"

            # DEMO_2: 0 + n = n - 2 ECHECS AVANT SUCCES
            if "0 + n = n" in theorem or "zero_add" in theorem:
                if n == 0:
                    state_mgr.log_tactic_attempt("rfl", goal, 0.3, "Tentative naive")
                    state_mgr.designate_next_agent("VerifierAgent")
                    return "[TacticAgent] Tentative 1: rfl (devrait echouer). -> VerifierAgent"
                elif n == 1:
                    state_mgr.log_tactic_attempt("simp", goal, 0.4, "Simplification")
                    state_mgr.designate_next_agent("VerifierAgent")
                    return "[TacticAgent] Tentative 2: simp (insuffisant). -> VerifierAgent"
                else:
                    state_mgr.log_tactic_attempt("exact Nat.zero_add n", goal, 0.95, "Lemme exact")
                    state_mgr.designate_next_agent("VerifierAgent")
                    return "[TacticAgent] Tactique: exact Nat.zero_add n. -> VerifierAgent"

            # DEMO_3: a * c + b * c = (a + b) * c - 4 ECHECS AVANT SUCCES
            if "a * c + b * c" in theorem or "add_mul_distrib" in theorem:
                tactics_sequence = [
                    ("rfl", 0.2, "Tentative naive"),
                    ("simp", 0.3, "Simplification basique"),
                    ("ring", 0.4, "Solveur arithmetique"),
                    ("rw [Nat.add_mul]", 0.5, "Réécriture partielle"),
                    ("rw [<- Nat.add_mul]", 0.95, "Forme correcte de distributivité")
                ]
                if n < len(tactics_sequence):
                    tactic, conf, desc = tactics_sequence[n]
                    state_mgr.log_tactic_attempt(tactic, goal, conf, desc)
                    state_mgr.designate_next_agent("VerifierAgent")
                    return f"[TacticAgent] Tentative {n+1}: {tactic}. -> VerifierAgent"
                else:
                    state_mgr.log_tactic_attempt("rw [<- Nat.add_mul]", goal, 0.95, "Solution")
                    state_mgr.designate_next_agent("VerifierAgent")
                    return "[TacticAgent] Tactique finale: rw [<- Nat.add_mul]. -> VerifierAgent"

            # DEMO_4: m * n = n * m - Exploration par induction (8-10 iterations)
            if "m * n = n * m" in theorem or "mul_comm_manual" in theorem:
                tactics_sequence = [
                    ("rfl", 0.1, "Tentative naive"),
                    ("simp", 0.15, "Simplification"),
                    ("ring", 0.2, "Ring ne suffit pas pour axiomes"),
                    ("omega", 0.2, "Omega: arithmétique linéaire seulement"),
                    ("induction m", 0.4, "Induction sur m"),
                    ("exact Nat.mul_zero n", 0.5, "Cas de base"),
                    ("simp [Nat.succ_mul]", 0.6, "Cas inductif étape 1"),
                    ("rw [ih]", 0.7, "Utiliser hypothèse induction"),
                    ("simp [Nat.mul_succ]", 0.8, "Transformation finale"),
                    ("exact Nat.mul_comm m n", 0.95, "Lemme direct")
                ]
                if n < len(tactics_sequence):
                    tactic, conf, desc = tactics_sequence[n]
                    state_mgr.log_tactic_attempt(tactic, goal, conf, desc)
                    state_mgr.designate_next_agent("VerifierAgent")
                    return f"[TacticAgent] Tentative {n+1}: {tactic}. -> VerifierAgent"
                else:
                    state_mgr.log_tactic_attempt("exact Nat.mul_comm m n", goal, 0.95, "Solution")
                    state_mgr.designate_next_agent("VerifierAgent")
                    return "[TacticAgent] Tactique finale: exact Nat.mul_comm. -> VerifierAgent"

        # --- MODE GÉNÉRIQUE: Stratégie adaptative ---
        # Stratégie: essayer les tactiques simples d'abord
        if n == 0:
            tactic = "rfl"
            desc = "Reflexivite"
        elif n == 1:
            tactic = "simp"
            desc = "Simplification"
        elif n == 2:
            # Utiliser les lemmes trouvés
            if state.lemmas_found:
                lemma = state.lemmas_found[0].name if hasattr(state.lemmas_found[0], 'name') else str(state.lemmas_found[0])
                tactic = f"exact {lemma}"
                desc = "Lemme exact"
            else:
                tactic = "ring"
                desc = "Ring solver"
        elif n == 3:
            tactic = "omega"
            desc = "Arithmétique linéaire"
        elif n == 4:
            tactic = "linarith"
            desc = "Arithmétique linéaire avancée"
        else:
            tactic = "sorry"
            desc = "Abandon (preuve incomplète)"

        state_mgr.log_tactic_attempt(tactic, goal, 0.3 + n * 0.1, desc)
        state_mgr.designate_next_agent("VerifierAgent")
        return f"[TacticAgent] Tactique: {tactic}. -> VerifierAgent"

    # =========================================================================
    # VERIFIER AGENT
    # =========================================================================
    def _do_verify(self, state: ProofState, theorem: str, goal: str) -> str:
        """Vérification de la preuve."""
        state_mgr = self.plugins.get("state")
        if not state_mgr or not state.tactics_history:
            return "[VerifierAgent] Rien a verifier."

        last = state.tactics_history[-1]
        n = len(state.tactics_history)
        attempt_id = f"attempt_{n}"

        # --- MODE DEMO: Vérification hardcodée ---
        if USE_DEMO_MODE:
            # DEMO_1: n = n - rfl réussit immédiatement
            if "rfl" in last.tactic and "n = n" in theorem and "demo_rfl" in theorem:
                state_mgr.add_verification_result(attempt_id, True, "OK", "", "", 50.0)
                state_mgr.set_proof_complete(last.tactic)
                return f"[VerifierAgent] SUCCES! Preuve par reflexivite: {last.tactic}"

            # DEMO_2: 0 + n = n - succès à la 3ème tentative
            if "0 + n = n" in theorem or "zero_add" in theorem:
                if n < 3:
                    state_mgr.add_verification_result(attempt_id, False, f"Tentative {n}", "echec", "", 80.0)
                    state_mgr.designate_next_agent("CriticAgent")
                    return f"[VerifierAgent] Echec tentative {n}. -> CriticAgent"
                else:
                    state_mgr.add_verification_result(attempt_id, True, "OK", "", "", 100.0)
                    state_mgr.set_proof_complete(last.tactic)
                    return f"[VerifierAgent] SUCCES apres {n} tentatives! {last.tactic}"

            # DEMO_3: distributivité - succès à la 5ème tentative
            if "a * c + b * c" in theorem or "add_mul_distrib" in theorem:
                if n < 5:
                    state_mgr.add_verification_result(attempt_id, False, f"{n}/5", "continue", "", 100.0)
                    state_mgr.designate_next_agent("CriticAgent")
                    return f"[VerifierAgent] Etape {n}/5. -> CriticAgent"
                else:
                    state_mgr.add_verification_result(attempt_id, True, "OK", "", "", 150.0)
                    state_mgr.set_proof_complete(last.tactic)
                    return f"[VerifierAgent] SUCCES! Distributivite prouvee apres {n} etapes."

            # DEMO_4: mul_comm - succès à la 10ème tentative
            if "m * n = n * m" in theorem or "mul_comm_manual" in theorem:
                if n < 10:
                    state_mgr.add_verification_result(attempt_id, False, f"{n}/10", "continue", "", 120.0)
                    state_mgr.designate_next_agent("CriticAgent")
                    return f"[VerifierAgent] Etape {n}/10. -> CriticAgent"
                else:
                    state_mgr.add_verification_result(attempt_id, True, "OK", "", "", 200.0)
                    state_mgr.set_proof_complete(last.tactic)
                    return f"[VerifierAgent] SUCCES! Commutativite prouvee apres {n} iterations."

        # --- MODE GÉNÉRIQUE: Vérification simulée ---
        # Simuler la vérification (en mode réel, appeler Lean)
        success_tactics = ["rfl", "simp", "ring", "omega", "linarith", "exact"]
        is_success = any(t in last.tactic for t in success_tactics) and n >= 2

        if is_success or "sorry" in last.tactic:
            state_mgr.add_verification_result(attempt_id, True, "OK", "", "", 100.0)
            state_mgr.set_proof_complete(last.tactic)
            return f"[VerifierAgent] SUCCES! {last.tactic}"
        else:
            state_mgr.add_verification_result(attempt_id, False, goal, "echec", "", 50.0)
            state_mgr.designate_next_agent("CriticAgent")
            return f"[VerifierAgent] Echec: {last.tactic}. -> CriticAgent"

    # =========================================================================
    # CRITIC AGENT
    # =========================================================================
    def _do_critic(self, state: ProofState, theorem: str) -> str:
        """Analyse critique et feedback."""
        state_mgr = self.plugins.get("state")
        if not state_mgr:
            return "[CriticAgent] Plugin manquant."

        n = len(state.tactics_history)

        # --- MODE DEMO: Feedback pédagogique ---
        if USE_DEMO_MODE:
            if "0 + n = n" in theorem or "zero_add" in theorem:
                if n == 1:
                    state_mgr.designate_next_agent("TacticAgent")
                    return "[CriticAgent] rfl echoue car 0+n n'est pas syntaxiquement n. Essayer simp. -> TacticAgent"
                else:
                    state_mgr.designate_next_agent("TacticAgent")
                    return "[CriticAgent] simp ne suffit pas. Utiliser le lemme Nat.zero_add directement. -> TacticAgent"

            if "a * c + b * c" in theorem or "add_mul_distrib" in theorem:
                hints = [
                    "rfl echoue: pas d'égalité syntaxique.",
                    "simp ne connait pas cette forme de distributivité.",
                    "ring ne peut pas gérer les Nat directement.",
                    "Essayer rw avec Nat.add_mul dans le bon sens."
                ]
                hint = hints[min(n-1, len(hints)-1)]
                state_mgr.designate_next_agent("TacticAgent")
                return f"[CriticAgent] {hint} -> TacticAgent"

            if "m * n = n * m" in theorem or "mul_comm_manual" in theorem:
                if n < 5:
                    state_mgr.designate_next_agent("TacticAgent")
                    return f"[CriticAgent] Tentative {n} echouee. Explorer l'induction. -> TacticAgent"
                else:
                    state_mgr.designate_next_agent("CoordinatorAgent")
                    return "[CriticAgent] Besoin de coordination pour stratégie induction. -> CoordinatorAgent"

        # --- MODE GÉNÉRIQUE ---
        state_mgr.designate_next_agent("TacticAgent")
        return f"[CriticAgent] Tentative {n} echouee. Essayer une autre approche. -> TacticAgent"

    # =========================================================================
    # COORDINATOR AGENT
    # =========================================================================
    def _do_coordinate(self, state: ProofState, theorem: str) -> str:
        """Coordination de la stratégie globale."""
        state_mgr = self.plugins.get("state")
        if not state_mgr:
            return "[CoordinatorAgent] Plugin manquant."

        # --- MODE DEMO ---
        if USE_DEMO_MODE:
            if "a * c + b * c" in theorem or "add_mul_distrib" in theorem:
                state_mgr.set_proof_strategy("distributivity_rewrite")
                state_mgr.designate_next_agent("TacticAgent")
                return "[CoordinatorAgent] Strategie: réécriture avec Nat.add_mul. -> TacticAgent"

            if "m * n = n * m" in theorem or "mul_comm_manual" in theorem:
                state_mgr.set_proof_strategy("induction_then_lemma")
                state_mgr.designate_next_agent("TacticAgent")
                return "[CoordinatorAgent] Strategie: induction puis lemme direct. -> TacticAgent"

        # --- MODE GÉNÉRIQUE ---
        if "*" in theorem and "+" in theorem:
            state_mgr.set_proof_strategy("ring_solver")
            state_mgr.designate_next_agent("TacticAgent")
            return "[CoordinatorAgent] Strategie: ring solver pour arithmétique. -> TacticAgent"

        if theorem.count("+") >= 2 or theorem.count("*") >= 2:
            state_mgr.set_proof_strategy("ac_normalization")
            state_mgr.designate_next_agent("TacticAgent")
            return "[CoordinatorAgent] Strategie: AC normalization. -> TacticAgent"

        state_mgr.designate_next_agent("TacticAgent")
        return "[CoordinatorAgent] Strategie par defaut. -> TacticAgent"

    # =========================================================================
    # LLM MODE (Appels réels à OpenAI)
    # =========================================================================
    def _call_llm(self, message: str, state: ProofState) -> str:
        """Appelle le LLM OpenAI avec function calling."""
        state_summary = json.dumps(state.get_state_snapshot(summarize=True), indent=2)
        tools = self._build_openai_tools()

        nl = chr(10)
        user_content = f"ETAT ACTUEL:{nl}{state_summary}{nl}{nl}TACHE:{nl}{message}"
        messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": user_content}
        ]

        max_tool_calls = 10
        tool_results = []

        for iteration in range(max_tool_calls):
            try:
                model = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-5.6-sol")
                use_mct = any(model.startswith(p) for p in ('gpt-4.5', 'gpt-5', 'o1', 'o3'))
                token_param = {"max_completion_tokens": 1000} if use_mct else {"max_tokens": 1000}

                response = self._openai_client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=tools if tools else None,
                    tool_choice="auto" if tools else None,
                    temperature=0.3,
                    **token_param
                )

                assistant_message = response.choices[0].message

                if assistant_message.tool_calls:
                    messages.append(assistant_message.model_dump())

                    for tool_call in assistant_message.tool_calls:
                        func_name = tool_call.function.name
                        try:
                            arguments = json.loads(tool_call.function.arguments)
                        except json.JSONDecodeError:
                            arguments = {}

                        result = self._execute_tool_call(func_name, arguments)
                        tool_results.append(func_name.split("__")[-1])

                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": result
                        })
                else:
                    final_response = assistant_message.content or "(pas de reponse)"
                    if tool_results:
                        actions = ", ".join(tool_results[:5])
                        final_response = f"Actions: {actions}{nl}{final_response}"
                    return f"[{self.name}] {final_response}"

            except Exception as e:
                return f"[{self.name}] Erreur LLM: {e}"

        actions = ", ".join(tool_results[:5])
        return f"[{self.name}] Max tool calls. Actions: {actions}"

print("SimpleAgent: classe initialisee")


SimpleAgent: classe initialisee


## 2.2. Patterns de Delegation Multi-Agents

Les instructions ci-dessus définissent les **règles de delegation** entre agents :

| Agent | Rôle | Delegue vers |
|-------|------|-------------|
| **SearchAgent** | Recherche lemmes Mathlib | TacticAgent (si lemmes trouves) |
| **TacticAgent** | Genere tactiques Lean | VerifierAgent (toujours) |
| **VerifierAgent** | Vérifie preuve formelle | CriticAgent (si echec) / COMPLETE (si succes) |
| **CriticAgent** | Analyse erreurs | SearchAgent (retry) / CoordinatorAgent (si bloque) |
| **CoordinatorAgent** | Re-orchestre stratégie | SearchAgent (nouvelle stratégie) |

**Flow nominal** (preuve simple) :
```
SearchAgent → TacticAgent → VerifierAgent → COMPLETE
```

**Flow avec echec** (preuve complexe) :
```
SearchAgent → TacticAgent → VerifierAgent (FAIL)
   ↓
CriticAgent analyse erreur
   ↓
   +-- Erreur simple → SearchAgent (retry avec nouvelles contraintes)
   +-- Erreur complexe → CoordinatorAgent (changement stratégie)
```

**Note critique** : dans l'execution committée de ce notebook (mode LLM reel), `CriticAgent` intervient apres chaque echec de verification - tours 5 et 8 dans DEMO_2, tours 5, 7, 10 et 13 dans DEMO_3, tour 5 dans DEMO_4 - mais `CoordinatorAgent` n'est pas active dans ce run : chaque erreur est corrigee sans changement de strategie globale. DEMO_4 (`pow_add_manual` : `a ^ (m + n) = a ^ m * a ^ n`), concu comme stress-test d'induction, reste le candidat naturel pour declencher la re-orchestration sur un run ou le lemme direct `Nat.pow_add` n'est pas trouve d'emblee.


### 2.3. Quand CriticAgent et CoordinatorAgent Interviennent

#### CriticAgent : Analyse d'Echecs de Tactiques

**Declenche par VerifierAgent quand** :
- `verify_proof()` retourne `success=False`
- Erreur Lean detectee : type mismatch, tactic failed, unknown identifier
- Preuve incomplète après application de tactique

**Responsabilites** :
1. Parser l'erreur Lean (extraire type, message, contexte)
2. Identifier la cause (lemme incorrect, tactique inadequate, goal mal compris)
3. Proposer correction :
   - Erreur simple (lemme manquant) → Delegue SearchAgent avec contraintes
   - Erreur complexe (stratégie incorrecte) → Delegue CoordinatorAgent

**Exemple d'intervention** :
```
[Tour 5] VerifierAgent: FAIL - "type mismatch, expected Nat but got Bool"
[Tour 6] CriticAgent: "TacticAgent a applique 'exact lemma_bool' mais goal attend Nat.
                       SearchAgent doit chercher lemmes avec type Nat -> Nat."
[Tour 7] SearchAgent: Recherche lemmes type-aware...
```

**Pourquoi si rare dans les demos actuelles** (mesures du run committé) :
- DEMO_1 : aucune erreur, `rfl` direct - aucun agent critique solicite
- DEMO_2 : deux interventions CriticAgent (tours 5, 8) avant `exact (Nat.right_distrib a b c).symm`
- DEMO_3 : quatre interventions CriticAgent (tours 5, 7, 10, 13) - la preuve finale est une induction complete
- DEMO_4 : une intervention CriticAgent (tour 5), puis resolution par le lemme direct `Nat.pow_add`
- CoordinatorAgent : pas active dans ce run - les echecs observes sont corriges sans changement de strategie

#### CoordinatorAgent : Re-Orchestration Stratégique

**Declenche par CriticAgent quand** :
- Echecs multiples consécutifs (3+ itérations sans progres)
- Stratégie actuelle bloquee (EXPLORATION → REFINEMENT → toujours FAIL)
- Pattern d'erreur complexe (induction necessaire mais pas tentee)

**Responsabilites** :
1. Analyser l'historique complet (`proof_state.tactic_history`)
2. Identifier pattern d'echec (loop, stratégie inadequate)
3. Changer stratégie globale :
   - EXPLORATION → VALIDATION (essayer preuves directes)
   - REFINEMENT → RECOVERY (backtrack + nouvelle approche)
4. Reset partiel de ProofState (clear failed tactics, keep lemmas)

**Exemple d'intervention** :
```
[Tour 8] CriticAgent: "Echec 3x consécutif avec même lemme. Stratégie bloquee."
[Tour 9] CoordinatorAgent: "Detection pattern: goal necessite induction mais pas tentee.
                            Changement stratégie: EXPLORATION → RECOVERY.
                            Ajout contrainte: TacticAgent DOIT considerer 'induction'."
[Tour 10] SearchAgent: Recherche lemmes inductifs...
```

**Pourquoi pas active dans le run committé** :
- Les echecs observes (DEMO_2, DEMO_3, DEMO_4) sont corriges au niveau CriticAgent sans changement de strategie
- Aucune serie de 3+ echecs consecutifs assez riche pour forcer la re-orchestration
- DEMO_4 (`pow_add_manual`) **PEUT** declencher la re-orchestration sur un autre run (mode LLM non deterministe) si :
  - TacticAgent essaie `rw` ou `simp` sans trouver le lemme direct → echecs repetes
  - CriticAgent detecte le besoin d'induction
  - CoordinatorAgent change stratégie vers RECOVERY

#### Activation des Agents Critiques

| Scénario | SearchAgent | TacticAgent | VerifierAgent | CriticAgent | CoordinatorAgent |
|----------|-------------|-------------|---------------|-------------|------------------|
| **Preuve triviale** (rfl) | ✗ | ✓ | ✓ | ✗ | ✗ |
| **Lemme direct trouve** (exact) | ✓ | ✓ | ✓ | ✗ | ✗ |
| **Lemme incorrect** (type mismatch) | ✓ | ✓ | ✓ | ✓ | ✗ |
| **Tactique echoue 1x** (retry) | ✓ | ✓ | ✓ | ✓ | ✗ |
| **Tactique echoue 3x** (bloque) | ✓ | ✓ | ✓ | ✓ | ✓ |
| **Induction necessaire** | ✓ | ✓ | ✓ | ✓ | ✓ |

**Conclusion** : Pour tester CriticAgent et CoordinatorAgent, nous devons utiliser des théorèmes ou :
1. Mathlib n'a PAS de lemme direct exact match
2. Preuve necessite composition de tactiques (rw + simp + induction)
3. Première tentative echoue et necessite correction

**DEMO_4 (`pow_add_manual`) est concu pour ca** - mais seulement en mode LLM, ou la generation de tactiques peut echouer plusieurs fois avant de trouver l'induction.


In [9]:

# =============================================================================
# Factory pour creer les agents (SK ou fallback)
# =============================================================================

def create_agents(
    plugins: Dict[str, Any],
    state: ProofState,
    use_sk: bool = True,
    use_simulation: bool = False
) -> Dict[str, Any]:
    """
    Cree les 5 agents specialises.

    Args:
        plugins: Dictionnaire des plugins SK
        state: État partage de la preuve
        use_sk: Utiliser Semantic Kernel si disponible
        use_simulation: Mode simulation (sans appels LLM)

    Returns:
        Dictionnaire {nom_agent: agent}
    """
    if use_sk and SK_AVAILABLE and not use_simulation:
        return _create_sk_agents(plugins, state)
    else:
        return _create_simple_agents(plugins, use_simulation)


def _create_simple_agents(plugins: Dict[str, Any], use_simulation: bool) -> Dict[str, Any]:
    """Cree les agents en mode fallback/simulation."""
    return {
        "SearchAgent": SimpleAgent("SearchAgent", SEARCH_AGENT_INSTRUCTIONS, plugins, use_simulation),
        "TacticAgent": SimpleAgent("TacticAgent", TACTIC_AGENT_INSTRUCTIONS, plugins, use_simulation),
        "VerifierAgent": SimpleAgent("VerifierAgent", VERIFIER_AGENT_INSTRUCTIONS, plugins, use_simulation),
        "CriticAgent": SimpleAgent("CriticAgent", CRITIC_AGENT_INSTRUCTIONS, plugins, use_simulation),
        "CoordinatorAgent": SimpleAgent("CoordinatorAgent", COORDINATOR_AGENT_INSTRUCTIONS, plugins, use_simulation),
    }


def _create_llm_service(service_name: str):
    """
    Cree le service LLM en fonction du provider configure.

    Args:
        service_name: Nom du service ("OpenAI", "Anthropic", "OpenRouter")

    Returns:
        Tuple (service, service_id, model_name)
    """
    service_name = service_name.lower()

    if service_name == "anthropic":
        # Anthropic Claude
        if not ANTHROPIC_AVAILABLE:
            raise ImportError("Anthropic connector not available. Install: pip install semantic-kernel[anthropic]")
        api_key = os.getenv("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY not set in environment")
        model = os.getenv("ANTHROPIC_CHAT_MODEL_ID", "claude-sonnet-4-5")
        print(f"[LLM Provider] Anthropic - Model: {model}")
        service = AnthropicChatCompletion(
            service_id="anthropic",
            ai_model_id=model,
            api_key=api_key
        )
        return service, "anthropic", model

    elif service_name == "openrouter":
        # OpenRouter (OpenAI-compatible API)
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise ValueError("OPENROUTER_API_KEY not set in environment")
        base_url = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
        model = os.getenv("OPENROUTER_CHAT_MODEL_ID", "anthropic/claude-sonnet-4")
        print(f"[LLM Provider] OpenRouter - Model: {model} - Base URL: {base_url}")
        from openai import AsyncOpenAI
        client = AsyncOpenAI(api_key=api_key, base_url=base_url)
        service = OpenAIChatCompletion(
            service_id="openrouter",
            ai_model_id=model,
            async_client=client
        )
        return service, "openrouter", model

    else:
        # OpenAI (default)
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not set in environment")
        model = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-5.2")
        print(f"[LLM Provider] OpenAI - Model: {model}")
        service = OpenAIChatCompletion(
            service_id="openai",
            ai_model_id=model,
            api_key=api_key
        )
        return service, "openai", model


def _create_sk_agents(plugins: Dict[str, Any], state: ProofState) -> Dict[str, Any]:
    """
    Cree les agents avec Semantic Kernel ChatCompletionAgent.

    Utilise:
    - Le service LLM configure via GLOBAL_LLM_SERVICE (OpenAI, Anthropic, OpenRouter)
    - FunctionChoiceBehavior.Auto() pour le function calling automatique
    - Les plugins existants sont passes aux agents

    Configuration via .env:
        GLOBAL_LLM_SERVICE: "OpenAI" | "Anthropic" | "OpenRouter" (defaut: "Anthropic")

    Pour OpenAI:
        OPENAI_API_KEY, OPENAI_CHAT_MODEL_ID

    Pour Anthropic:
        ANTHROPIC_API_KEY, ANTHROPIC_CHAT_MODEL_ID

    Pour OpenRouter:
        OPENROUTER_API_KEY, OPENROUTER_BASE_URL, OPENROUTER_CHAT_MODEL_ID
    """
    # Creer le kernel
    kernel = Kernel()

    # Determiner le provider LLM (defaut: Anthropic car quota OpenAI epuise)
    llm_service_name = os.getenv("GLOBAL_LLM_SERVICE", "Anthropic")
    print(f"\n{'='*60}")
    print(f"Configuration LLM Service: {llm_service_name}")
    print(f"{'='*60}")

    # Creer le service LLM
    service, service_id, model = _create_llm_service(llm_service_name)
    kernel.add_service(service)

    # Ajouter les plugins au kernel
    for plugin_name, plugin in plugins.items():
        kernel.add_plugin(plugin, plugin_name=plugin_name)

    # Configuration pour auto function calling
    settings = kernel.get_prompt_execution_settings_from_service_id(service_id=service_id)
    settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

    # Creer les agents
    agents = {}
    agent_configs = [
        ("SearchAgent", SEARCH_AGENT_INSTRUCTIONS),
        ("TacticAgent", TACTIC_AGENT_INSTRUCTIONS),
        ("VerifierAgent", VERIFIER_AGENT_INSTRUCTIONS),
        ("CriticAgent", CRITIC_AGENT_INSTRUCTIONS),
        ("CoordinatorAgent", COORDINATOR_AGENT_INSTRUCTIONS),
    ]

    for name, instructions in agent_configs:
        agents[name] = ChatCompletionAgent(
            kernel=kernel,
            name=name,
            instructions=instructions,
            arguments=KernelArguments(settings=settings)
        )

    print(f"\nCrees {len(agents)} agents SK avec provider {llm_service_name} et modele {model}")
    return agents


# =============================================================================
# Test des Agents
# =============================================================================

print("\n=== Test des Agents ===")

# Creer l'environnement
test_state = ProofState(
    theorem_statement="theorem add_zero (n : Nat) : n + 0 = n",
    current_goal="n + 0 = n"
)
runner = LeanRunner(backend="subprocess", timeout=30)

# Creer les plugins
plugins = {
    "state": ProofStateManagerPlugin(test_state),
    "search": LeanSearchPlugin(runner),
    "tactic": LeanTacticPlugin(),
    "verification": LeanVerificationPlugin(runner)
}

# Determiner le mode de fonctionnement
# Vérifier si au moins un provider LLM est configure
llm_service = os.getenv("GLOBAL_LLM_SERVICE", "Anthropic").lower()
has_api_key = False

if llm_service == "openai":
    has_api_key = bool(os.getenv("OPENAI_API_KEY"))
elif llm_service == "anthropic":
    has_api_key = bool(os.getenv("ANTHROPIC_API_KEY"))
elif llm_service == "openrouter":
    has_api_key = bool(os.getenv("OPENROUTER_API_KEY"))

USE_SK = SK_AVAILABLE and has_api_key
USE_SIMULATION = not USE_SK  # Simulation si SK non disponible ou pas de cle API

if USE_SK:
    print(f"Mode: Semantic Kernel avec provider {os.getenv('GLOBAL_LLM_SERVICE', 'Anthropic')}")
else:
    print(f"Mode: Simulation (SK_AVAILABLE={SK_AVAILABLE}, has_api_key={has_api_key})")

# Creer les agents
agents = create_agents(plugins, test_state, use_sk=USE_SK, use_simulation=USE_SIMULATION)

# Test rapide en mode simulation
if USE_SIMULATION:
    print("\nTest SearchAgent (simulation):")
    response = agents["SearchAgent"].invoke("Trouve des lemmes pour n + 0 = n", test_state)
    print(response)
    print(f"Etat apres SearchAgent:\n{test_state}")



=== Test des Agents ===


Mode: Semantic Kernel avec provider Anthropic

Configuration LLM Service: Anthropic
[LLM Provider] Anthropic - Model: claude-sonnet-4-5



Crees 5 agents SK avec provider Anthropic et modele claude-sonnet-4-5


### 2.4. Vue d'Ensemble des 5 Agents Specialises

La fonction `create_agents()` instancie les 5 agents avec :
- **Instructions** : Prompts systemiques definissant rôle et règles de delegation
- **Plugins** : Fonctions exposees (search, tactic generation, vérification, etc.)
- **Modèle LLM** : gpt-5.2 (ou simulation si mode LLM desactive)

#### Signatures des agents

```python
SearchAgent(
    plugins=[lean_search_plugin, state_plugin],
    instructions="Trouve lemmes Mathlib pertinents..."
)

TacticAgent(
    plugins=[tactic_plugin, state_plugin],
    instructions="Genere tactiques Lean avec confiance..."
)

VerifierAgent(
    plugins=[verification_plugin, state_plugin],
    instructions="Compile et vérifie preuves formelles..."
)

CriticAgent(
    plugins=[state_plugin],
    instructions="Analyse echecs et propose corrections..."
)

CoordinatorAgent(
    plugins=[state_plugin],
    instructions="Re-orchestre stratégie globale..."
)
```

**Pattern cle** : Chaque agent n'a acces qu'aux plugins dont il a besoin (principe de moindre privilege). Le `state_plugin` est partage par tous pour consulter/modifier ProofState.


## 3. Orchestration Multi-Agents

L'orchestration determine comment les agents sont selectionnes et quand la conversation se termine.

**DelegatingSelectionStrategy** (Pattern recommande):
- Chaque agent designe explicitement le suivant via `designate_next_agent()`
- Si aucune designation, utilise un agent par defaut (CoordinatorAgent)

**ProofCompleteTermination**:
- Termine si `proof_complete == True`
- Termine si `iteration_count >= max_iterations`

### 3.1. Workflow Complet de Preuve

Cette demonstration montre le workflow multi-agents complet:
1. **CoordinatorAgent** initialise la session
2. **SearchAgent** recherche les lemmes pertinents
3. **TacticAgent** propose des tactiques
4. **VerifierAgent** vérifie avec Lean
5. **CriticAgent** intervient en cas d'echec

## 🎭 Orchestration Multi-Agents

### Le problème de l'orchestration

Avec 5 agents, qui parle quand ? Deux approches :

1. **Statique** : SearchAgent → TacticAgent → VerifierAgent (toujours)
   - Simple mais rigide
   - Pas de backtracking

2. **Dynamique** : Décisions basées sur l'état de la preuve
   - Flexible mais complexe
   - Permet le backtracking et la récupération d'erreur

**Nous utilisons l'approche dynamique.**

### Stratégies d'orchestration

#### ProofSelectionStrategy

Décide **quel agent agit** à chaque tour :

```python
class ProofSelectionStrategy:
    def select_next_agent(self, state: ProofState, agents: List[str]) -> str:
        if state.phase == ProofPhase.INIT:
            return "CoordinatorAgent"
        elif state.phase == ProofPhase.SEARCH:
            return "SearchAgent"
        # ...
```

## ProofTerminationStrategy

Décide **quand arrêter** la session :

```python
class ProofTerminationStrategy:
    def should_terminate(self, state: ProofState, itération: int) -> Tuple[bool, str]:
        if state.phase == ProofPhase.COMPLETE:
            return (True, "Preuve complète!")
        if itération >= max_iterations:
            return (True, "Timeout atteint")
        # ...
```

## Boucle principale

```python
while not should_terminate:
    # 1. Sélectionner agent
    agent_name = selection_strategy.select_next_agent(state, agents)

    # 2. Exécuter agent (appelle le LLM)
    response = agent.chat(f"Phase: {state.phase}, Goal: {state.current_goal}")

    # 3. L'agent appelle des plugins (modifie l'état)
    # Exemple: log_tactic_attempt("rw [Nat.add_zero]")

    # 4. Mettre à jour phase selon résultat
    update_phase(state)

    # 5. Vérifier condition de terminaison
    should_terminate, reason = termination_strategy.should_terminate(state, itération)
```

## Snapshots : Observer l'orchestration

À chaque tour, on sauvegarde :

```json
{
  "itération": 5,
  "agent": "TacticAgent",
  "phase_before": "SEARCH",
  "phase_after": "TACTIC_GEN",
  "action": "Generated tactic: rw [Nat.add_zero]",
  "state_snapshot": {...}
}
```

**Utilité** : Voir exactement quelle décision chaque agent a prise.

In [10]:
# =============================================================================
# Section 3 - Strategies d'Orchestration (Pattern Argument_Analysis)
# =============================================================================
# Strategies personnalisees basees sur l'état partage (pas sur l'historique)

# Fix for Jupyter event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

import logging
from typing import Dict, Any, List, Optional

print("Imports OK : nest_asyncio, logging, typing")


Imports OK : nest_asyncio, logging, typing


## 3.2. ProofSelectionStrategy : Sélection d'Agent : Sélection d'Agent Basee sur l'État

**Pattern inspire de Argument_Analysis** : Sélection d'agent via **designation explicite** dans ProofState.

### Architecture

```python
class ProofSelectionStrategy(SelectionStrategy):
    async def next(agents, history) -> Agent:
        # 1. Lire designation explicite
        designated = state.consume_next_agent_designation()
        
        # 2. Si designation presente, utiliser cet agent
        if designated:
            return agents_map[designated]
        
        # 3. Sinon, utiliser agent par defaut (SearchAgent)
        return agents_map[default_agent_name]
```

### Différence avec Semantic Kernel Standard

| Semantic Kernel Standard | ProofSelectionStrategy (Custom) |
|-------------------------|--------------------------------|
| Parse historique messages | Lit `state.next_agent_designation` |
| Sélection basee sur keywords | Sélection basee sur état partage |
| Stateless (pas de mémoire) | Stateful (ProofState) |
| Complexite O(n) messages | Complexite O(1) |

**Avantage** : Chaque agent designe explicitement son successeur via `state.designate_next_agent()`, evitant parsing d'historique fragile.


In [11]:
# =============================================================================
# ProofSelectionStrategy - Selection basee sur l'état partage
# =============================================================================
# NOTE: Ces classes SK ne sont definies que si SK est disponible.
# Le mode simulation n'en a pas besoin.

if SK_AVAILABLE:
    from semantic_kernel.agents.strategies.selection.selection_strategy import SelectionStrategy

    class ProofSelectionStrategy(SelectionStrategy):
        """Strategie de selection SK (non utilisee en mode simulation)."""
        pass
    print("ProofSelectionStrategy: classe definie (SK disponible)")
else:
    print("ProofSelectionStrategy: Skipped (SK non disponible)")


ProofSelectionStrategy: classe definie (SK disponible)


## 3.3. ProofTerminationStrategy : Detection de Completion : Detection de Completion

**Responsabilite** : Detecter quand arreter l'orchestration multi-agents.

### Critères de Terminaison

```python
class ProofTerminationStrategy(TerminationStrategy):
    async def should_terminate(agents, history) -> bool:
        # 1. Preuve complète detectee
        if state.proof_complete:
            return True
        
        # 2. Max itérations atteint
        if state.current_iteration >= max_iterations:
            return True
        
        # 3. Timeout (optionnel)
        if time.time() - start_time > timeout:
            return True
        
        return False
```

### Comparaison avec Autres Patterns

| Pattern | Terminaison basee sur | Avantages | Inconvenients |
|---------|----------------------|-----------|---------------|
| **Message-based** | Keyword dans dernier message ("DONE", "COMPLETE") | Simple, standard SK | Fragile, depend du LLM |
| **State-based** (ce notebook) | `state.proof_complete` flag | Robuste, déterministe | Necessite état partage |
| **Itération-based** | Compteur max itérations | Toujours termine | Peut stopper preuve incomplète |
| **Consensus-based** | Vote agents (majorite) | Robuste aux erreurs | Complexe, lent |

**Notre choix** : Combinaison **state-based + itération-based** pour garantir terminaison.


In [12]:
# =============================================================================
# ProofTerminationStrategy - Terminaison basee sur l'état partage
# =============================================================================
# NOTE: Ces classes SK ne sont definies que si SK est disponible.
# Le mode simulation n'en a pas besoin.

if SK_AVAILABLE:
    from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy

    class ProofTerminationStrategy(TerminationStrategy):
        """Strategie de terminaison SK (non utilisee en mode simulation)."""
        pass
    print("ProofTerminationStrategy: classe definie (SK disponible)")
else:
    print("ProofTerminationStrategy: Skipped (SK non disponible)")


ProofTerminationStrategy: classe definie (SK disponible)


### 3.4. ProofAgentGroupChat : Chat Multi-Agents : Orchestration Multi-Agents

**Classe orchestrateur** pour gérer la conversation multi-agents avec Semantic Kernel.

#### Architecture

```python
class ProofAgentGroupChat:
    def __init__(agents, state, use_sk=True):
        self.agents = agents  # Dict[str, ChatCompletionAgent]
        self.state = state    # ProofState partage
    
    def run(initial_message, verbose=True) -> str:
        # Exécute conversation multi-agents
        if use_sk:
            return await _run_sk(...)  # Semantic Kernel
        else:
            return _run_fallback(...)   # Simulation
```

**Pattern clé** :
- Utilise `ProofSelectionStrategy` pour sélectionner agents
- Utilise `ProofTerminationStrategy` pour détecter fin
- Crée `AgentGroupChat` de Semantic Kernel avec ces stratégies
- Fallback en mode simulation si SK non disponible


In [13]:
class ProofAgentGroupChat:
    """
    Orchestre les agents pour la preuve de théorèmes.
    Supporte mode simulation (SimpleAgent) et mode SK (ChatCompletionAgent).
    """

    def __init__(self, agents: Dict[str, Any], state: ProofState, use_sk: bool = True):
        self.agents = agents
        self.state = state
        self.use_sk = use_sk and SK_AVAILABLE
        self.history = []
        self._proof_tactics_found = []  # Track tactics found across iterations

    def run(self, initial_message: str, verbose: bool = True) -> str:
        """Exécute la conversation multi-agents."""
        if self.use_sk:
            # Mode Semantic Kernel - utilise async
            import asyncio
            import nest_asyncio
            nest_asyncio.apply()
            try:
                loop = asyncio.get_event_loop()
                return loop.run_until_complete(self._run_sk(initial_message, verbose))
            except RuntimeError:
                return asyncio.run(self._run_sk(initial_message, verbose))
        else:
            # Mode simulation - sync
            return self._run_fallback(initial_message, verbose)

    async def _run_sk(self, initial_message: str, verbose: bool = True) -> str:
        """Exécution avec Semantic Kernel ChatCompletionAgent - LOGGING AMELIORE."""
        from semantic_kernel.contents.chat_history import ChatHistory
        from semantic_kernel.contents.chat_message_content import ChatMessageContent
        from semantic_kernel.contents.utils.author_role import AuthorRole
        from datetime import datetime
        import re

        def clean_response(text: str) -> str:
            """Nettoie les reponses LLM (supprime newlines excessifs)."""
            # Remplacer sequences de 3+ newlines par 2
            text = re.sub(r'\n{3,}', '\n\n', text)
            # Supprimer espaces en debut/fin
            text = text.strip()
            return text

        def format_timestamp() -> str:
            """Retourne timestamp lisible."""
            return datetime.now().strftime("%H:%M:%S.%f")[:-3]

        if verbose:
            session_start = datetime.now()
            print("=" * 70)
            print(f"SESSION MULTI-AGENTS (SK)")
            print(f"Theoreme: {initial_message[:100]}...")
            print("=" * 70)

        # Creer l'historique de chat partage entre les agents
        chat_history = ChatHistory()
        chat_history.add_user_message(initial_message)

        current_message = initial_message
        agent_order = ["SearchAgent", "TacticAgent", "VerifierAgent", "CriticAgent", "CoordinatorAgent"]

        for i in range(self.state.max_iterations):
            self.state.iteration = i + 1
            self.state.increment_iteration()
            iter_start = datetime.now()

            # Determiner l'agent a utiliser
            designated = self.state.consume_next_agent_designation()
            if designated and designated in self.agents:
                agent_name = designated
            else:
                agent_name = agent_order[i % len(agent_order)]

            agent = self.agents.get(agent_name)
            if not agent:
                continue

            if verbose:
                print(f"\n{'─' * 70}")
                elapsed = (datetime.now() - session_start).total_seconds()
                print(f"[+{elapsed:.2f}s] TOUR {self.state.iteration_count} | Agent: {agent_name}")
                print(f"{'─' * 70}")

            # Invoquer l'agent SK de maniere asynchrone
            try:
                response_text = ""

                # ChatCompletionAgent.invoke() prend un ChatHistory et retourne un AsyncIterable
                async for response in agent.invoke(chat_history):
                    if hasattr(response, 'content') and response.content:
                        response_text += str(response.content)
                    elif hasattr(response, 'items'):
                        # Si c'est un ChatMessageContent avec items
                        for item in response.items:
                            if hasattr(item, 'text'):
                                response_text += item.text

                if not response_text:
                    response_text = f"[{agent_name}] Pas de reponse"

                # Nettoyer la reponse
                response_text = clean_response(response_text)

                # Ajouter la reponse a l'historique
                chat_history.add_assistant_message(response_text)

                # Ajouter le prochain message utilisateur (contexte pour le prochain agent)
                if i < self.state.max_iterations - 1:
                    next_context = f"Continue la preuve. Reponse precedente de {agent_name}: {response_text[:200]}"
                    chat_history.add_user_message(next_context)

                # Mettre a jour l'état selon la reponse
                self._update_state_from_response(agent_name, response_text)

            except Exception as e:
                import traceback
                response_text = f"Erreur agent {agent_name}: {str(e)}"
                if verbose:
                    print(f"  [ERROR] {e}")
                    traceback.print_exc()

            iter_duration = (datetime.now() - iter_start).total_seconds()

            self.history.append({
                "iteration": self.state.iteration_count,
                "agent": agent_name,
                "response": response_text,
                "duration_s": iter_duration
            })

            if verbose:
                # Afficher reponse complète (pas tronquee)
                print(f"  Reponse ({len(response_text)} chars, {iter_duration:.2f}s):")
                # Indenter chaque ligne pour lisibilite
                for line in response_text.split('\n')[:30]:  # Max 30 lignes
                    if line.strip():
                        print(f"    {line}")
                if response_text.count('\n') > 30:
                    print(f"    ... ({response_text.count(chr(10)) - 30} lignes supprimees)")

            if self.state.proof_complete:
                if verbose:
                    elapsed = (datetime.now() - session_start).total_seconds(); print(f"\n[+{elapsed:.2f}s] PREUVE TROUVEE!")
                    print(f"  Tactique finale: {self.state.final_proof}")
                break

            current_message = response_text

        if verbose:
            print("\n" + "=" * 70)
            total_time = (datetime.now() - session_start).total_seconds()
            print(f"SESSION TERMINEE (duree totale: {total_time:.2f}s)")
            print(f"  Iterations: {self.state.iteration_count}")
            print(f"  Lemmes decouverts: {len(self.state.discovered_lemmas)}")
            print(f"  Tactiques essayees: {len(self.state.tactics_history)}")
            print("=" * 70)

        return self.state.final_proof or "Preuve non trouvee"


    def _update_state_from_response(self, agent_name: str, response: str):
        """Met a jour l'état partage en fonction de la reponse de l'agent."""
        import re
        response_lower = response.lower()

        # Detection des lemmes decouverts
        if "lemma:" in response_lower or "found:" in response_lower or "nat." in response_lower:
            lemma_matches = re.findall(r'(Nat\.\w+|Eq\.\w+|List\.\w+)', response)
            for lemma in lemma_matches:
                if lemma not in self.state.discovered_lemmas:
                    self.state.discovered_lemmas.append(lemma)

        # Detection des tactiques - track across iterations
        proof_patterns = [
            (r'simp\s*\[[^\]]*\]', 'simp'),
            (r'\brfl\b', 'rfl'),
            (r'exact\s+\w+', 'exact'),
            (r'\bring\b', 'ring'),
            (r'\bomega\b', 'omega'),
            (r'\blinarith\b', 'linarith'),
            (r'\bdecide\b', 'decide'),
        ]

        for pattern, tactic_name in proof_patterns:
            if re.search(pattern, response, re.IGNORECASE):
                if tactic_name not in self._proof_tactics_found:
                    self._proof_tactics_found.append(tactic_name)
                    self.state.tactics_history.append(response[:100])

        # Detection de preuve complète (audit #17357, constats 1-3) :
        # ANTI-ARTEFACT -- une preuve n'est déclarée complète que si
        # l'extraction retient un énoncé COMPLET : "theorem ... := by
        # <tactique>" avec au moins un token après by, ou une forme terminale
        # ":= <tactique>" (ex. rfl). Les signaux verbaux ("verified", "preuve
        # réussie"...), un bloc tronqué à ":= by" ou un nom de tactique isolé
        # ("rfl", "exact") ne suffisent plus : jamais de faux "Success: True"
        # sur un artefact de parsing.

        # Extraction validée : parcourir TOUS les blocs de code de la réponse
        # (le modèle énonce souvent le théorème dans un premier bloc et la
        # preuve complète dans un second) et retenir le premier énoncé complet.
        validated_proof = None
        for block_match in re.finditer(r'```(?:lean)?\s*\n(.*?)```', response, re.DOTALL):
            code_content = block_match.group(1).strip()
            is_complete = re.search(r':=\s*by\s+\S+', code_content) or re.search(r':=\s*rfl\b', code_content)
            if is_complete:
                if len(code_content) > 400:
                    cut = code_content[:400]
                    nl = cut.rfind('\n')
                    code_content = cut[:nl] if nl > 0 else cut
                validated_proof = code_content
                break

        # Repli hors bloc de code : un énoncé complet sur une seule ligne.
        # Scan ligne par ligne -- jamais au-dela de la fin de ligne : un
        # énoncé tronqué à ":= by" ne peut pas être complété par la suite.
        if validated_proof is None:
            for line in response.split('\n'):
                m = re.search(r'theorem\s+\w+[^\n]*', line)
                if m and ':=' in m.group(0):
                    stmt = m.group(0).strip()
                    if re.search(r':=\s*by\s+\S+', stmt) or re.search(r':=\s*rfl\b', stmt):
                        validated_proof = stmt[:400]
                        break

        if validated_proof is not None:
            self.state.phase = ProofPhase.COMPLETE
            self.state.final_proof = validated_proof
        # Sinon : signaux insuffisants -- la boucle multi-agents continue,
        # l'echec d'extraction n'est jamais maquille en succes.

        # Detection de delegation
        delegate_patterns = [
            (r'@TacticAgent|delegate.*TacticAgent', 'TacticAgent'),
            (r'@VerifierAgent|delegate.*VerifierAgent', 'VerifierAgent'),
            (r'@CriticAgent|delegate.*CriticAgent', 'CriticAgent'),
            (r'@CoordinatorAgent|delegate.*CoordinatorAgent', 'CoordinatorAgent'),
            (r'@SearchAgent|delegate.*SearchAgent', 'SearchAgent'),
        ]
        for pattern, target in delegate_patterns:
            if re.search(pattern, response, re.IGNORECASE):
                self.state.designate_next_agent(target)
                break

    def _run_fallback(self, initial_message: str, verbose: bool = True) -> str:
        """Exécution sans Semantic Kernel (mode simulation) - LOGGING AMELIORE."""
        from datetime import datetime

        def format_timestamp() -> str:
            return datetime.now().strftime("%H:%M:%S.%f")[:-3]

        session_start = datetime.now()
        if verbose:
            print("=" * 70)
            print(f"SESSION MULTI-AGENTS (Simulation)")
            print(f"Theoreme: {initial_message[:100]}...")
            print("=" * 70)

        current_message = initial_message
        agent_order = ["SearchAgent", "TacticAgent", "VerifierAgent", "CriticAgent", "CoordinatorAgent"]

        for i in range(self.state.max_iterations):
            self.state.iteration = i + 1
            iter_start = datetime.now()

            designated = self.state.consume_next_agent_designation()
            if designated and designated in self.agents:
                agent_name = designated
            else:
                agent_name = agent_order[i % len(agent_order)]

            agent = self.agents.get(agent_name)
            if not agent:
                continue

            if verbose:
                print(f"\n{'─' * 70}")
                elapsed = (datetime.now() - session_start).total_seconds()
                print(f"[+{elapsed:.2f}s] TOUR {self.state.iteration_count} | Agent: {agent_name}")
                print(f"{'─' * 70}")

            response = agent.invoke(current_message, self.state)
            iter_duration = (datetime.now() - iter_start).total_seconds()

            self.history.append({
                "iteration": self.state.iteration_count,
                "agent": agent_name,
                "response": response,
                "duration_s": iter_duration
            })

            if verbose:
                print(f"  Reponse ({len(response)} chars, {iter_duration:.3f}s):")
                for line in response.split('\n')[:20]:
                    if line.strip():
                        print(f"    {line}")
                if response.count('\n') > 20:
                    print(f"    ... ({response.count(chr(10)) - 20} lignes supprimees)")

            if self.state.proof_complete:
                if verbose:
                    elapsed = (datetime.now() - session_start).total_seconds(); print(f"\n[+{elapsed:.2f}s] PREUVE TROUVEE!")
                    print(f"  Tactique finale: {self.state.final_proof}")
                break

            current_message = response

        if verbose:
            print("\n" + "=" * 70)
            total_time = (datetime.now() - session_start).total_seconds()
            print(f"SESSION TERMINEE (duree totale: {total_time:.2f}s)")
            print(f"  Iterations: {self.state.iteration_count}")
            print(f"  Lemmes decouverts: {len(self.state.discovered_lemmas)}")
            print(f"  Tactiques essayees: {len(self.state.tactics_history)}")
            print("=" * 70)

        return self.state.final_proof or "Preuve non trouvee"

print("Classe ProofAgentGroupChat definie")


Classe ProofAgentGroupChat definie


## 3.5. Test des Stratégies

Code de test pour valider :
- **ProofTerminationStrategy** : Détecte `state.proof_complete`
- **SimpleOrchestratorAgent** : Exécute conversation avec désignation d'agents

**Exécution automatique** lors du chargement de la cellule.


In [14]:
# Test des Strategies
# =============================================================================

print("=== Test des Strategies ===")

test_state = ProofState(
    theorem_statement="theorem test (n : Nat) : n = n",
    current_goal="n = n",
    max_iterations=5
)

print(f"State cree: {test_state.session_id}")
print(f"Phase initiale: {test_state.phase.value}")

# Test designation
test_state.designate_next_agent("TacticAgent")
designated = test_state.consume_next_agent_designation()
print(f"Designation test: {designated}")

# Test proof_complete
print(f"proof_complete initial: {test_state.proof_complete}")
test_state.phase = ProofPhase.COMPLETE
print(f"proof_complete apres COMPLETE: {test_state.proof_complete}")

print("\nStrategies pretes pour utilisation avec AgentGroupChat")


=== Test des Strategies ===
State cree: ce3c78ea
Phase initiale: init
Designation test: TacticAgent
proof_complete initial: False
proof_complete apres COMPLETE: True

Strategies pretes pour utilisation avec AgentGroupChat


## 4. Demonstrations Progressives

Les 4 demonstrations suivantes illustrent le fonctionnement du système multi-agents avec une progression de complexite croissante :

| Demo | Théorème | Complexite | Itérations | Technique |
|------|----------|------------|------------|-----------|
| DEMO_1 | `n = n` | Triviale | 2 | `rfl` direct |
| DEMO_2 | `a*c + b*c = (a+b)*c` | Simple | 5 | Recherche + reecriture inversee |
| DEMO_3 | `m * n = n * m` | Intermediaire | 8 | Lemme `Nat.mul_comm` |
| DEMO_4 | `a^(m+n) = a^m * a^n` | Avancee | 15 | Induction + lemmes multiples |

**Progression pedagogique** :
- DEMO_1 : Validation du pipeline (cas trivial)
- DEMO_2 : Introduction de la recherche de lemmes
- DEMO_3 : Propriété fondamentale avec `exact`
- DEMO_4 : Stress-test avec induction complexe


In [15]:
# =============================================================================
# Configuration du mode d'exécution
# =============================================================================

# Mode LLM ou Simulation - MODIFIEZ ICI selon vos besoins
USE_LLM_MODE = True  # True = appels LLM réels, False = simulation

# Les DEMOs suivantes utilisent des définitions inline pour iteration independante.
# Chaque DEMO peut etre executee et corrigee independamment.

print("=" * 70)
print("DEMONSTRATIONS PROGRESSIVES - SYSTEME MULTI-AGENTS")
print("=" * 70)
print(f"Mode: {'LLM (OpenAI)' if USE_LLM_MODE else 'Simulation'}")
print("Les 4 DEMOs testent des theoremes de complexite croissante:")
print("  DEMO_1: Reflexivite (rfl)")
print("  DEMO_2: Recherche de lemme (Nat.zero_add)")
print("  DEMO_3: Reecriture inversee (add_mul)")
print("  DEMO_4: Induction double (mul_comm)")
print("=" * 70)


DEMONSTRATIONS PROGRESSIVES - SYSTEME MULTI-AGENTS
Mode: LLM (OpenAI)
Les 4 DEMOs testent des theoremes de complexite croissante:
  DEMO_1: Reflexivite (rfl)
  DEMO_2: Recherche de lemme (Nat.zero_add)
  DEMO_3: Reecriture inversee (add_mul)
  DEMO_4: Induction double (mul_comm)


Demonstration complète du système multi-agents avec orchestration des différents rôles et exécution de bout en bout.

In [16]:
# =============================================================================
# Section 4 - Demonstration Complète
# =============================================================================

def prove_with_multi_agents(
    theorem: str,
    goal: str = "",
    max_iterations: int = 20,
    verbose: bool = True,
    use_simulation: bool = None  # None = auto-detect
) -> Dict[str, Any]:
    """
    Prouve un théorème en utilisant le système multi-agents.

    Args:
        theorem: L'enonce du théorème complet
        goal: Le but a prouver (extrait du théorème si non fourni)
        max_iterations: Nombre maximum d'iterations
        verbose: Afficher les logs
        use_simulation: True=simulation, False=LLM réel, None=auto

    Returns:
        Dict avec résultats et metriques
    """
    import time
    start_time = time.time()

    # Auto-detection du mode
    if use_simulation is None:
        api_key = os.getenv("OPENAI_API_KEY", "")
        has_valid_key = api_key and len(api_key) > 10 and not api_key.startswith("sk-...")
        use_simulation = not has_valid_key

    # 1. Creer l'état
    if not goal:
        if ":" in theorem:
            goal = theorem.split(":")[-1].strip()

    state = ProofState(
        theorem_statement=theorem,
        current_goal=goal,
        max_iterations=max_iterations
    )

    # 2. Creer le runner Lean
    runner = LeanRunner(backend="subprocess", timeout=30)

    # 3. Creer les plugins
    plugins = {
        "state": ProofStateManagerPlugin(state),
        "search": LeanSearchPlugin(runner),
        "tactic": LeanTacticPlugin(),
        "verification": LeanVerificationPlugin(runner)
    }

    # 4. Creer les agents
    use_sk = SK_AVAILABLE and not use_simulation
    agents = create_agents(plugins, state, use_sk=use_sk, use_simulation=use_simulation)

    # 5. Configurer les strategies
    # Strategies gerees automatiquement par ProofAgentGroupChat

    # 6. Creer le groupe de chat
    chat = ProofAgentGroupChat(
        agents=agents,
        state=state,
        use_sk=use_sk
    )

    mode_str = "Semantic Kernel" if use_sk else ("Simulation" if use_simulation else "OpenAI direct")
    if verbose:
        print(f"Mode: {mode_str}")

    # 7. Exécuter
    result = chat.run(f"Prouver: {theorem}", verbose=verbose)

    # 8. Collecter les metriques
    elapsed = time.time() - start_time
    metrics = {
        "success": state.proof_complete,
        "theorem": theorem,
        "final_proof": state.final_proof,
        "iterations": state.iteration_count,
        "lemmas_discovered": len(state.discovered_lemmas),
        "tactics_tried": len(state.tactics_history),
        "verifications": len(state.verification_results),
        "total_time_s": round(elapsed, 2),
        "lean_time_ms": round(state.total_lean_time_ms, 2),
        "mode": mode_str
    }

    return metrics


# =============================================================================
# Test de la demonstration
# =============================================================================

print("\n" + "=" * 60)
print("DEMONSTRATION MULTI-AGENTS POUR THEOREM PROVING")
print("=" * 60)

# =============================================================================
# Section 4 - Démonstrations Progressives Multi-Agents
# =============================================================================

# Configuration



DEMONSTRATION MULTI-AGENTS POUR THEOREM PROVING


### 4.1. DEMO_1 : Reflexivite Pure

**Objectif** : Valider le pipeline complet avec un théorème trivial

**Théorème** : `theorem demo_rfl (n : Nat) : n = n`

**Comportement attendu** : 1-2 itérations : `rfl` suffit immediatement


In [17]:
# =============================================================================
# DEMO_1 : REFLEXIVITY
# =============================================================================

# Définition inline pour iteration independante
demo_1 = {
    "name": "DEMO_1_REFLEXIVITY",
    "theorem": "theorem demo_rfl (n : Nat) : n = n",
    "expected_iterations": 2,
    "expected_lemmas": 0,
    "complexity": "Triviale - rfl suffit",
    "strategy": "rfl"
}

print("\n" + "=" * 70)
print(f"DEMO 1/4: {demo_1['name']}")
print("=" * 70)
print(f"Theoreme: {demo_1['theorem']}")
print(f"Complexite: {demo_1['complexity']}")
print(f"Iterations attendues: {demo_1['expected_iterations']}")
print("=" * 70)

result_1 = prove_with_multi_agents(
    theorem=demo_1["theorem"],
    max_iterations=20,
    verbose=True,
    use_simulation=not USE_LLM_MODE
)

print(f"\nResultat DEMO_1:")
print(f"  - Success: {result_1['success']}")
print(f"  - Iterations: {result_1['iterations']} (attendu: {demo_1['expected_iterations']})")
print(f"  - Proof: {result_1['final_proof']}")


DEMO 1/4: DEMO_1_REFLEXIVITY
Theoreme: theorem demo_rfl (n : Nat) : n = n
Complexite: Triviale - rfl suffit
Iterations attendues: 2



Configuration LLM Service: Anthropic
[LLM Provider] Anthropic - Model: claude-sonnet-4-5



Crees 5 agents SK avec provider Anthropic et modele claude-sonnet-4-5
Mode: Semantic Kernel
SESSION MULTI-AGENTS (SK)
Theoreme: Prouver: theorem demo_rfl (n : Nat) : n = n...

──────────────────────────────────────────────────────────────────────
[+0.00s] TOUR 2 | Agent: SearchAgent
──────────────────────────────────────────────────────────────────────


  Reponse (450 chars, 27.82s):
    Excellent ! J'ai trouvé les lemmes clés pour prouver ce théorème :
    **Lemmes découverts :**
    1. **`rfl`** : preuve de réflexivité `a = a` - directement applicable
    2. **`Eq.refl`** : preuve explicite de réflexivité
    **Analyse :**
    Ce théorème `n = n` est un cas classique de réflexivité de l'égalité. La tactique `rfl` devrait suffire à le prouver directement.
    Je délègue maintenant à **TacticAgent** qui utilisera ces lemmes pour construire la preuve.

──────────────────────────────────────────────────────────────────────
[+27.82s] TOUR 3 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (302 chars, 12.12s):
    **Tactique proposée : `rfl`**
    **Justification :**
    - But actuel : `n = n` 
    - C'est un cas de réflexivité pure de l'égalité
    - `rfl` est conçue exactement pour ce type de preuve
    - SearchAgent a confirmé que `rfl` est disponible
    Je délègue à **VerifierAgent** pour vérifier si cette tactique fonctionne !

──────────────────────────────────────────────────────────────────────
[+39.95s] TOUR 4 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


  Reponse (522 chars, 22.41s):
    # ✅ **PREUVE COMPLÈTE ET VÉRIFIÉE !**
    **Résultat de vérification :**
    - ✅ **Succès** : La preuve compile sans erreur
    - ⏱️ **Temps d'exécution** : 9543.39 ms
    - 🎯 **Aucun but restant** : La preuve est complète
    **Code final :**
    ```lean
    theorem demo_rfl (n : Nat) : n = n := by rfl
    ```
    **Explication :**
    La tactique `rfl` (réflexivité) a résolu directement le but `n = n`, car l'égalité est réflexive par définition en Lean 4. C'est la preuve la plus simple et élégante pour ce théorème.
    🎉 **Preuve terminée avec succès !**

[+62.36s] PREUVE TROUVEE!
  Tactique finale: theorem demo_rfl (n : Nat) : n = n := by rfl

SESSION TERMINEE (duree totale: 62.36s)
  Iterations: 4
  Lemmes decouverts: 2
  Tactiques essayees: 2

Resultat DEMO_1:
  - Success: True
  - Iterations: 4 (attendu: 2)
  - Proof: theorem demo_rfl (n : Nat) : n = n := by rfl


### 4.2. DEMO_2 : Distributivite Inversee

**Objectif** : Montrer la recherche de lemme avec reecriture inversee

**Théorème** : `theorem add_mul_distrib (a b c : Nat) : a * c + b * c = (a + b) * c`

**Comportement attendu** : 4-6 itérations : recherche `Nat.add_mul`, reecriture inversee


In [18]:
# =============================================================================
# DEMO_2 : DISTRIBUTIVITY
# =============================================================================

# Définition inline pour iteration independante
demo_2 = {
    "name": "DEMO_2_DISTRIBUTIVITY",
    "theorem": "theorem add_mul_distrib (a b c : Nat) : a * c + b * c = (a + b) * c",
    "expected_iterations": 5,
    "expected_lemmas": 1,
    "complexity": "Simple - forme inversee du lemme standard",
    "strategy": "rw [<- Nat.add_mul]"
}

print("\n" + "=" * 70)
print(f"DEMO 2/4: {demo_2['name']}")
print("=" * 70)
print(f"Theoreme: {demo_2['theorem']}")
print(f"Complexite: {demo_2['complexity']}")
print(f"Iterations attendues: {demo_2['expected_iterations']}")
print("=" * 70)

result_2 = prove_with_multi_agents(
    theorem=demo_2["theorem"],
    max_iterations=20,
    verbose=True,
    use_simulation=not USE_LLM_MODE
)

print(f"\nResultat DEMO_2:")
print(f"  - Success: {result_2['success']}")
print(f"  - Iterations: {result_2['iterations']} (attendu: {demo_2['expected_iterations']})")
print(f"  - Proof: {result_2['final_proof']}")


DEMO 2/4: DEMO_2_DISTRIBUTIVITY
Theoreme: theorem add_mul_distrib (a b c : Nat) : a * c + b * c = (a + b) * c
Complexite: Simple - forme inversee du lemme standard
Iterations attendues: 5



Configuration LLM Service: Anthropic
[LLM Provider] Anthropic - Model: claude-sonnet-4-5



Crees 5 agents SK avec provider Anthropic et modele claude-sonnet-4-5
Mode: Semantic Kernel
SESSION MULTI-AGENTS (SK)
Theoreme: Prouver: theorem add_mul_distrib (a b c : Nat) : a * c + b * c = (a + b) * c...

──────────────────────────────────────────────────────────────────────
[+0.00s] TOUR 2 | Agent: SearchAgent
──────────────────────────────────────────────────────────────────────


  Reponse (382 chars, 62.91s):
    **Résumé de ma recherche :**
    J'ai identifié les lemmes essentiels pour prouver `a * c + b * c = (a + b) * c` :
    - **Nat.right_distrib** donne directement `(a + b) * c = a * c + b * c`, donc on peut l'utiliser avec `symm` ou `rw`
    - **Nat.add_comm** permet de commuter les additions si nécessaire
    TacticAgent peut maintenant construire la preuve en utilisant ces lemmes découverts !

──────────────────────────────────────────────────────────────────────
[+62.91s] TOUR 3 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (446 chars, 11.86s):
    **J'ai proposé ma PREMIÈRE tentative : `rfl`**
    Selon ma stratégie d'exploration systématique, je commence toujours par les tactiques les plus simples. C'est peu probable de fonctionner ici car `a * c + b * c` et `(a + b) * c` ne sont pas définitionnellement égaux, mais c'est important de l'essayer pour montrer le processus complet de découverte.
    **VerifierAgent** va maintenant tester cette tactique et nous dira si elle fonctionne ou non ! 🎯

──────────────────────────────────────────────────────────────────────
[+74.76s] TOUR 4 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


...

  Reponse (527 chars, 11.73s):
    **TROISIÈME TENTATIVE : `exact Nat.right_distrib.symm`**
    Comme suggéré par **CriticAgent**, j'utilise maintenant le lemme **Nat.right_distrib** découvert par **SearchAgent**.
    **Pourquoi cette tactique ?**
    - `Nat.right_distrib` prouve : `(a + b) * c = a * c + b * c`
    - Notre but est : `a * c + b * c = (a + b) * c`
    - Avec `.symm`, on inverse l'égalité pour matcher exactement notre but
    - La tactique `exact` ferme le but si le terme a exactement le bon type
    **VerifierAgent**, à toi de vérifier si cette preuve fonctionne ! 🎯

──────────────────────────────────────────────────────────────────────
[+120.26s] TOUR 7 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


  Reponse (636 chars, 24.46s):
    **❌ ÉCHEC DE VÉRIFICATION**
    **Erreur :** `Unknown constant 'Nat.right_distrib.symm'`
    **Diagnostic :**
    - La tactique `exact Nat.right_distrib.symm` ne fonctionne pas
    - Le lemme `Nat.right_distrib` existe mais `.symm` ne peut pas être appliqué directement dans le mode `exact`
    **J'ai enregistré ce résultat et délègue à CriticAgent pour analyser l'erreur et suggérer une correction.**
    Le problème est probablement qu'il faut utiliser une approche différente, comme :
    - `rw [Nat.right_distrib]` pour la réécriture
    - Ou `symm; exact Nat.right_distrib` pour inverser d'abord le but
    **➡️ CriticAgent** : À toi d'analyser cette erreur ! 🔍

──────────────────────────────────────────────────────────────────────
[+144.73s] TOUR 8 | Agent: CriticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (140 chars, 12.43s):
    **➡️ TacticAgent :** Essaie `exact (Nat.right_distrib a b c).symm` avec les arguments explicites, ou simplement `rw [Nat.right_distrib]` ! 🎯

──────────────────────────────────────────────────────────────────────
[+157.15s] TOUR 9 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (678 chars, 12.35s):
    **QUATRIÈME TENTATIVE : `exact (Nat.right_distrib a b c).symm`**
    Comme suggéré par **CriticAgent**, j'utilise maintenant le lemme avec **les arguments explicites** !
    **Différence clé :**
    - ❌ `Nat.right_distrib.symm` → Lean ne sait pas à quels arguments appliquer `.symm`
    - ✅ `(Nat.right_distrib a b c).symm` → On applique d'abord le lemme à `a, b, c`, ce qui donne une preuve de `(a + b) * c = a * c + b * c`, puis `.symm` inverse cette égalité
    **Logique :**
    1. `Nat.right_distrib a b c` prouve : `(a + b) * c = a * c + b * c`
    2. `.symm` inverse en : `a * c + b * c = (a + b) * c`
    3. C'est exactement notre but ! 🎯
    **VerifierAgent**, vérifie si cette preuve est correcte ! 🚀

──────────────────────────────────────────────────────────────────────
[+169.51s] TOUR 10 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


  Reponse (819 chars, 20.07s):
    ---
    # ✅ **PREUVE COMPLÈTE ET VÉRIFIÉE !**
    ```lean
    theorem add_mul_distrib (a b c : Nat) : a * c + b * c = (a + b) * c := by
      exact (Nat.right_distrib a b c).symm
    ```
    **📊 Résumé de la vérification :**
    - ✅ **Succès** : La preuve compile sans erreur
    - ⏱️ **Temps d'exécution** : 4134 ms
    - 🎯 **Aucun but restant** : La preuve est complète
    **🔍 Explication de la solution :**
    1. Le lemme `Nat.right_distrib` de Mathlib prouve : `(a + b) * c = a * c + b * c`
    2. En appliquant `.symm`, on inverse l'égalité : `a * c + b * c = (a + b) * c`
    3. C'est exactement le but demandé !
    **🎓 Leçon apprise :**
    Quand on utilise `.symm` sur un lemme, il faut d'abord l'appliquer à ses arguments explicites : `(Nat.right_distrib a b c).symm` plutôt que `Nat.right_distrib.symm`.
    **La preuve est maintenant terminée et enregistrée ! 🎉**

[+189.58s] PREUVE TROUVEE!
  Tactique finale: theorem add_mul_distrib (a b c : Nat) : 

### 4.3. DEMO_3 : Commutativite Multiplication

**Objectif** : Tester une propriété fondamentale avec recherche

**Théorème** : `theorem mul_comm_manual (m n : Nat) : m * n = n * m`

**Comportement attendu** : 6-10 itérations : decouverte de `Nat.mul_comm`


In [19]:
# =============================================================================
# DEMO_3 : MUL_COMM
# =============================================================================

# Définition inline pour iteration independante
demo_3 = {
    "name": "DEMO_3_MUL_COMM",
    "theorem": "theorem mul_comm_manual (m n : Nat) : m * n = n * m",
    "expected_iterations": 8,
    "expected_lemmas": 2,
    "complexity": "Intermediaire - necessite lemme de commutativite",
    "strategy": "exact Nat.mul_comm m n"
}

print("\n" + "=" * 70)
print(f"DEMO 3/4: {demo_3['name']}")
print("=" * 70)
print(f"Theoreme: {demo_3['theorem']}")
print(f"Complexite: {demo_3['complexity']}")
print(f"Iterations attendues: {demo_3['expected_iterations']}")
print("=" * 70)

result_3 = prove_with_multi_agents(
    theorem=demo_3["theorem"],
    max_iterations=20,
    verbose=True,
    use_simulation=not USE_LLM_MODE
)

print(f"\nResultat DEMO_3:")
print(f"  - Success: {result_3['success']}")
print(f"  - Iterations: {result_3['iterations']} (attendu: {demo_3['expected_iterations']})")
print(f"  - Proof: {result_3['final_proof']}")


DEMO 3/4: DEMO_3_MUL_COMM
Theoreme: theorem mul_comm_manual (m n : Nat) : m * n = n * m
Complexite: Intermediaire - necessite lemme de commutativite
Iterations attendues: 8



Configuration LLM Service: Anthropic
[LLM Provider] Anthropic - Model: claude-sonnet-4-5



Crees 5 agents SK avec provider Anthropic et modele claude-sonnet-4-5
Mode: Semantic Kernel
SESSION MULTI-AGENTS (SK)
Theoreme: Prouver: theorem mul_comm_manual (m n : Nat) : m * n = n * m...

──────────────────────────────────────────────────────────────────────
[+0.00s] TOUR 2 | Agent: SearchAgent
──────────────────────────────────────────────────────────────────────


  Reponse (597 chars, 55.81s):
    ## Résumé de ma recherche
    J'ai identifié les lemmes essentiels pour prouver `m * n = n * m` manuellement :
    **Lemmes trouvés :**
    - `Nat.mul_zero` : n * 0 = 0
    - `Nat.zero_mul` : 0 * n = 0  
    - `Nat.mul_succ` : n * m.succ = n * m + n
    - `Nat.succ_mul` : n.succ * m = n * m + m
    - `Nat.add_comm` : n + m = m + n
    - `Nat.add_assoc` : (n + m) + k = n + (m + k)
    **Stratégie suggérée :**
    La preuve devrait procéder par induction sur `n` (ou `m`), en utilisant les lemmes ci-dessus pour montrer les cas de base et inductif. TacticAgent peut maintenant utiliser ces lemmes pour construire la preuve complète.

──────────────────────────────────────────────────────────────────────
[+55.81s] TOUR 3 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (631 chars, 14.14s):
    ---
    ## Tactique proposée : `rfl`
    **Justification :** Selon ma stratégie d'exploration pédagogique, je commence TOUJOURS par les tactiques les plus simples (`rfl` ou `trivial`), même si je sais qu'elles échoueront probablement.
    **Attente :** Cette tactique échouera car `m * n` et `n * m` ne sont pas définitionnellement égaux en Lean. La commutativité de la multiplication nécessite une preuve par induction.
    **Prochaines étapes si échec :** 
    - Tentative 2 : `simp`
    - Tentative 3 : Utiliser les lemmes découverts par SearchAgent
    - Tentative 4 : Induction sur `n` ou `m`
    ➡️ **VerifierAgent**, à toi de vérifier cette tactique !

──────────────────────────────────────────────────────────────────────
[+69.95s] TOUR 4 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


...

  Reponse (28 chars, 15.66s):
    [CriticAgent] Pas de reponse

──────────────────────────────────────────────────────────────────────
[+224.47s] TOUR 14 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (100 chars, 11.97s):
    ➡️ **VerifierAgent**, vérifie cette tactique d'induction structurée avec traitement du cas de base !

──────────────────────────────────────────────────────────────────────
[+236.45s] TOUR 15 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


  Reponse (149 chars, 18.53s):
    ➡️ **TacticAgent**, remplace le `sorry` dans le cas `succ` par une preuve complète utilisant `ih`, `Nat.mul_succ`, `Nat.succ_mul` et `Nat.add_comm` !

──────────────────────────────────────────────────────────────────────
[+254.97s] TOUR 16 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (779 chars, 15.75s):
    ---
    ## Tactique proposée : Preuve complète par induction
    ```lean
    induction n with 
    | zero => simp [Nat.mul_zero, Nat.zero_mul] 
    | succ n ih => rw [Nat.mul_succ, Nat.succ_mul, ih, Nat.add_comm]
    ```
    **Justification :**
    1. **Cas de base (`zero`)** : `simp` avec `Nat.mul_zero` et `Nat.zero_mul` prouve `m * 0 = 0 * m`
    2. **Cas inductif (`succ n`)** : Avec l'hypothèse `ih : m * n = n * m`, on prouve `m * n.succ = n.succ * m` par réécritures successives :
       - `Nat.mul_succ` : `m * n.succ = m * n + m`
       - `Nat.succ_mul` : `n.succ * m = n * m + m`
       - `ih` : remplace `m * n` par `n * m`
       - `Nat.add_comm` : finalise l'égalité
    **Confiance : 95%** - Cette preuve suit exactement les lemmes identifiés par SearchAgent.
    ➡️ **VerifierAgent**, vérifie cette preuve complète !

──────────────────────────────────────────────────────────────────────
[+270.73s] TOUR 17 | Agent: SearchAgent
────────────────────

  Reponse (429 chars, 3.81s):
    ---
    ## ✅ Vérification RÉUSSIE !
    **Preuve complète validée :**
    ```lean
    theorem mul_comm_manual (m n : Nat) : m * n = n * m := by
      induction n with 
      | zero => simp [Nat.mul_zero, Nat.zero_mul] 
      | succ n ih => rw [Nat.mul_succ, Nat.succ_mul, ih, Nat.add_comm]
    ```
    **Résultat :** Aucune erreur, tous les buts résolus ✓
    **Preuve marquée comme complète dans l'état partagé.**
    ➡️ **CriticAgent**, valide cette preuve finale !

[+274.54s] PREUVE TROUVEE!
  Tactique finale: theorem mul_comm_manual (m n : Nat) : m * n = n * m := by
  induction n with 
  | zero => simp [Nat.mul_zero, Nat.zero_mul] 
  | succ n ih => rw [Nat.mul_succ, Nat.succ_mul, ih, Nat.add_comm]

SESSION TERMINEE (duree totale: 274.54s)
  Iterations: 17
  Lemmes decouverts: 12
  Tactiques essayees: 8

Resultat DEMO_3:
  - Success: True
  - Iterations: 17 (attendu: 8)
  - Proof: theorem mul_comm_manual (m n : Nat) : m * n = n * m := by
  induction n w

### 4.4. DEMO_4 : Addition des Puissances

**Objectif** : Stresser le système avec induction et lemmes multiples

**Théorème** : `theorem pow_add_manual (a m n : Nat) : a ^ (m + n) = a ^ m * a ^ n`

**Comportement attendu** : 12-18 itérations : induction sur n, decouverte de pow_succ, mul_assoc


In [20]:
# =============================================================================
# DEMO_4 : POWER_ADD
# =============================================================================

# Définition inline pour iteration independante
demo_4 = {
    "name": "DEMO_4_POWER_ADD",
    "theorem": "theorem pow_add_manual (a m n : Nat) : a ^ (m + n) = a ^ m * a ^ n",
    "expected_iterations": 15,
    "expected_lemmas": 4,
    "complexity": "Avancee - induction et lemmes auxiliaires multiples",
    "strategy": "induction n avec Nat.pow_succ, Nat.mul_assoc"
}

print("\n" + "=" * 70)
print(f"DEMO 4/4: {demo_4['name']}")
print("=" * 70)
print(f"Theoreme: {demo_4['theorem']}")
print(f"Complexite: {demo_4['complexity']}")
print(f"Iterations attendues: {demo_4['expected_iterations']}")
print("=" * 70)

result_4 = prove_with_multi_agents(
    theorem=demo_4["theorem"],
    max_iterations=20,
    verbose=True,
    use_simulation=not USE_LLM_MODE
)

print(f"\nResultat DEMO_4:")
print(f"  - Success: {result_4['success']}")
print(f"  - Iterations: {result_4['iterations']} (attendu: {demo_4['expected_iterations']})")
print(f"  - Proof: {result_4['final_proof']}")


DEMO 4/4: DEMO_4_POWER_ADD
Theoreme: theorem pow_add_manual (a m n : Nat) : a ^ (m + n) = a ^ m * a ^ n
Complexite: Avancee - induction et lemmes auxiliaires multiples
Iterations attendues: 15



Configuration LLM Service: Anthropic
[LLM Provider] Anthropic - Model: claude-sonnet-4-5



Crees 5 agents SK avec provider Anthropic et modele claude-sonnet-4-5
Mode: Semantic Kernel
SESSION MULTI-AGENTS (SK)
Theoreme: Prouver: theorem pow_add_manual (a m n : Nat) : a ^ (m + n) = a ^ m * a ^ n...

──────────────────────────────────────────────────────────────────────
[+0.00s] TOUR 2 | Agent: SearchAgent
──────────────────────────────────────────────────────────────────────


  Reponse (554 chars, 33.25s):
    J'ai terminé ma recherche de lemmes. J'ai identifié les lemmes essentiels pour prouver `a ^ (m + n) = a ^ m * a ^ n`:
    **Lemmes clés découverts:**
    1. `Nat.pow_succ`: définition récursive de la puissance
    2. `Nat.pow_zero`: cas de base
    3. `Nat.mul_assoc`: associativité de la multiplication
    4. `Nat.mul_one`: identité multiplicative
    5. `Nat.succ_add`: propriété de l'addition avec successeur
    **Stratégie suggérée:** Preuve par induction sur `n` avec ces lemmes.
    Je délègue maintenant à **TacticAgent** pour générer et appliquer les tactiques appropriées.

──────────────────────────────────────────────────────────────────────
[+33.25s] TOUR 3 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (390 chars, 12.41s):
    **TacticAgent** a proposé la tactique `rfl` comme première tentative exploratoire.
    **Justification:** Selon la stratégie d'exploration systématique, je dois d'abord essayer les tactiques les plus simples (`rfl`, `trivial`) avant de passer aux approches plus sophistiquées, même si je sais que cette propriété nécessitera probablement une preuve par induction.
    À toi, **VerifierAgent** ! 🎯

──────────────────────────────────────────────────────────────────────
[+45.66s] TOUR 4 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


...

  Reponse (28 chars, 12.91s):
    [CriticAgent] Pas de reponse

──────────────────────────────────────────────────────────────────────
[+75.31s] TOUR 6 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (254 chars, 12.18s):
    **Justification:** Je suis la stratégie systématique. `simp` pourrait potentiellement utiliser des lemmes de simplification automatiques de Mathlib, mais je m'attends à ce qu'il échoue aussi.
    À toi, **VerifierAgent** ! Vérifie si `simp` résout le but. 🎯

──────────────────────────────────────────────────────────────────────
[+87.49s] TOUR 7 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


  Reponse (30 chars, 15.71s):
    [VerifierAgent] Pas de reponse

──────────────────────────────────────────────────────────────────────
[+103.20s] TOUR 8 | Agent: TacticAgent
──────────────────────────────────────────────────────────────────────


  Reponse (248 chars, 13.33s):
    **TacticAgent** a proposé : `exact Nat.pow_add a m n`
    **Justification:** Je teste si Mathlib contient déjà ce lemme. Si oui, `exact` le résoudra directement. Sinon, je passerai aux tactiques structurelles (induction).
    À toi, **VerifierAgent** ! 🔍

──────────────────────────────────────────────────────────────────────
[+116.53s] TOUR 9 | Agent: VerifierAgent
──────────────────────────────────────────────────────────────────────


  Reponse (509 chars, 15.35s):
    ---
    🎉 **VerifierAgent - PREUVE COMPLÈTE !** ✅
    **Résultat:** SUCCÈS ✓
    La tactique `exact Nat.pow_add a m n` a résolu le théorème directement !
    **Preuve finale:**
    ```lean
    theorem pow_add_manual (a m n : Nat) : a ^ (m + n) = a ^ m * a ^ n := by
      exact Nat.pow_add a m n
    ```
    **Analyse:**
    - Le lemme `Nat.pow_add` existe déjà dans Mathlib
    - Il prouve exactement la même propriété
    - Temps d'exécution: 2348.57 ms
    - Aucune erreur
    **Conclusion:** La preuve est complète et vérifiée par le compilateur Lean 4. 🎊

[+131.88s] PREUVE TROUVEE!
  Tactique finale: theorem pow_add_manual (a m n : Nat) : a ^ (m + n) = a ^ m * a ^ n := by
  exact Nat.pow_add a m n

SESSION TERMINEE (duree totale: 131.88s)
  Iterations: 9
  Lemmes decouverts: 11
  Tactiques essayees: 5

Resultat DEMO_4:
  - Success: True
  - Iterations: 9 (attendu: 15)
  - Proof: theorem pow_add_manual (a m n : Nat) : a ^ (m + n) = a ^ m * a ^ n := by
 

### 4.5. Analyse Comparative des Résultats

**Objectif** : Comparer les résultats observes avec les attentes.

#### Resultats observes (mesures du run committe, mode LLM reel)

| Demo | Attendu | Observe | Preuve finale retenue |
|------|---------|---------|-----------------------|
| DEMO_1 | 2 iters | 4 | `theorem demo_rfl ... := by rfl` |
| DEMO_2 | 5 iters | 10 | `exact (Nat.right_distrib a b c).symm` |
| DEMO_3 | 8 iters | 17 | induction complete (`induction n with` + cas zero/succ) |
| DEMO_4 | 15 iters | 9 | `exact Nat.pow_add a m n` |

#### Lecture des ecarts

1. **DEMO_1, DEMO_2 et DEMO_3 depassent les attentes** : chaque echec de verification declenche CriticAgent (deux a quatre interventions par demo), la convergence demande plus de tours que prevu
2. **DEMO_4 en dessous de l'attente** : le lemme direct `Nat.pow_add` est trouve sans passer par l'induction prevue - Mathlib contient les lemmes exacts
3. **CoordinatorAgent pas active dans ce run** : les echecs sont corriges au niveau CriticAgent, sans changement de strategie globale

#### Extraction anti-artefact

La preuve finale n'est retenue par l'orchestrateur que si elle est **complete** : un enonce `theorem ... := by <tactique>` avec au moins un token apres `by`. Un bloc de code tronque a `:= by` ou un nom de tactique isole (`rfl`, `exact`) n'est jamais affiche comme succes.

#### Pour observer la re-orchestration (CoordinatorAgent)

```python
# Ce run est deja en mode LLM reel (USE_LLM_MODE = True)
# Un theoreme sans lemme direct dans Mathlib multiplierait les echecs
# consecutifs et deverrouillerait CoordinatorAgent (changement de strategie)
theorem_custom = "theorem custom_hard (n m k : Nat) : (n + m) * k = n * k + m * k + k"
```

**Conclusion** : ce run demontre l'*architecture* multi-agents en generation reelle, non-deterministe incluse : d'une execution a l'autre, DEMO_4 peut exigier l'induction (15+ iterations) ou tomber sur le lemme direct (9 iterations ici).

In [21]:
# =============================================================================
# Comparaison des Résultats
# =============================================================================

print("\n" + "=" * 70)
print("COMPARAISON DES RESULTATS")
print("=" * 70)

# Définitions inline pour iteration independante
demos_info = [
    {"name": "DEMO_1_REFLEXIVITY", "theorem": "n = n", "expected_iter": 2, "expected_lemmas": 0},
    {"name": "DEMO_2_DISTRIBUTIVITY", "theorem": "a*c + b*c = (a+b)*c", "expected_iter": 5, "expected_lemmas": 1},
    {"name": "DEMO_3_MUL_COMM", "theorem": "m * n = n * m", "expected_iter": 8, "expected_lemmas": 2},
    {"name": "DEMO_4_POWER_ADD", "theorem": "a^(m+n) = a^m * a^n", "expected_iter": 15, "expected_lemmas": 4},
]

results = [result_1, result_2, result_3, result_4]

print(f"{'Demo':<25} {'Success':<10} {'Iter':<12} {'Lemmas':<10} {'Status':<15}")
print("-" * 72)

for i, (demo, result) in enumerate(zip(demos_info, results), 1):
    success_str = "OK" if result["success"] else "FAILED"
    iter_str = f"{result['iterations']}/{demo['expected_iter']}"
    lemmas_str = f"{result.get('lemmas_discovered', 0)}/{demo['expected_lemmas']}"

    if result["success"]:
        if result["iterations"] <= demo["expected_iter"]:
            status = "Optimal"
        else:
            status = "Slow"
    else:
        status = "Failed"

    print(f"{demo['name']:<25} {success_str:<10} {iter_str:<12} {lemmas_str:<10} {status:<15}")

print("-" * 72)
total_success = sum(1 for r in results if r["success"])
print(f"Total: {total_success}/4 reussies")

# Resume de la progression
print("\n" + "=" * 70)
print("PROGRESSION DE COMPLEXITE")
print("=" * 70)
print("DEMO_1: Reflexivite     -> Pipeline validation (rfl)")
print("DEMO_2: Distributivite  -> Recherche + reecriture inversee")
print("DEMO_3: Commutativite   -> Propriete fondamentale (exact)")
print("DEMO_4: Puissances      -> Induction + lemmes multiples")


COMPARAISON DES RESULTATS
Demo                      Success    Iter         Lemmas     Status         
------------------------------------------------------------------------
DEMO_1_REFLEXIVITY        OK         4/2          2/0        Slow           
DEMO_2_DISTRIBUTIVITY     OK         10/5         5/1        Slow           
DEMO_3_MUL_COMM           OK         17/8         12/2       Slow           
DEMO_4_POWER_ADD          OK         9/15         11/4       Optimal        
------------------------------------------------------------------------
Total: 4/4 reussies

PROGRESSION DE COMPLEXITE
DEMO_1: Reflexivite     -> Pipeline validation (rfl)
DEMO_2: Distributivite  -> Recherche + reecriture inversee
DEMO_3: Commutativite   -> Propriete fondamentale (exact)
DEMO_4: Puissances      -> Induction + lemmes multiples


## Exercice 2 : Stratégie de sélection Round-Robin

Implementez une stratégie de sélection **Round-Robin** qui fait tourner les agents
de maniere cyclique (Agent1 -> Agent2 -> Agent3 -> Agent1 -> ...) independamment
de l'état de la preuve. C'est la stratégie la plus simple mais elle sert de base
de comparaison pour des stratégies plus intelligentes comme `PriorityCriticStrategy`.

**Objectif** : Implementer `RoundRobinStrategy.next()` qui cyclise parmi les agents.

**Étapes** :
1. Maintenir un compteur interne `self._index`
2. Dans `next()`, sélectionner `agents[self._index % len(agents)]`
3. Incrementer le compteur après chaque sélection

**Indice** : Initialisez `self._index = 0` dans `__init__`. Attention au cas `len(agents) == 0`.


In [22]:
# ============================================================
# Exercice 2 : RoundRobinSelectionStrategy
# ============================================================
# Votre objectif : implementer une strategie de selection
# qui fait tourner les agents de maniere cyclique.
#
# Rappel : ProofSelectionStrategy a la méthode :
#   async def next(self, agents: list, history: list) -> Agent
# ============================================================

try:
    class RoundRobinStrategy(ProofSelectionStrategy):
        """
        Strategie Round-Robin : selection cyclique des agents.
        Chaque appel a next() avance au prochain agent.
        """

        def __init__(self):
            super().__init__()
            # TODO: initialiser le compteur interne
            self._index = 0  # TODO étudiant

        async def next(self, agents, history):
            """Selectionne le prochain agent dans le cycle."""
            if not agents:
                raise ValueError("Aucun agent disponible")
            # TODO: selectionner agents[self._index % len(agents)]
            selected = agents[0]  # TODO étudiant
            # TODO: incrementer le compteur
            pass  # TODO étudiant
            return selected

    print("Exercice 2: RoundRobinStrategy - a completer")
except NameError:
    print("Exercice 2: RoundRobinStrategy - requiere Semantic Kernel (SK_AVAILABLE=True)")


Exercice 2: RoundRobinStrategy - a completer


## Exemple guide : Personnaliser la Stratégie d'Orchestration

### Objectif

Implementez une stratégie de sélection **prioritaire** : l'agent `CriticAgent` est toujours appele après un echec de `TacticAgent`,
même si la logique par defaut selectionnait un autre agent.

### Contexte

La classe `ProofSelectionStrategy` (section 3.2) determine quel agent parle a chaque tour.
La propriété `proof_state.error_count` indique le nombre de tactiques echouees.

### Consignes

1. Subclassez `ProofSelectionStrategy` en `PriorityCriticStrategy`
2. Surchargez `next(agents, history)` : si `proof_state.error_count > 0`, retourner `CriticAgent` en priorite
3. Sinon, deleguer a la logique parente (`super().next(...)`)
4. Testez votre stratégie en remplacant `ProofSelectionStrategy` dans `ProofAgentGroupChat`

**Hint** : La signature async est `async def next(self, agents, history) -> Agent`.
Acces a l'état : `self.proof_state.error_count` (entier >= 0, reference a stocker au constructeur).

In [23]:
# ============================================================
# Exemple guide : PriorityCriticStrategy
# ============================================================
# Votre objectif : implementer une strategie qui priorise
# CriticAgent apres chaque echec de tactique.
#
# Rappel : ProofSelectionStrategy a la méthode :
#   async def next(self, agents: list, history: list) -> Agent
#
# proof_state.error_count : nombre de tactiques echouees (int)
# ============================================================

try:
    class PriorityCriticStrategy(ProofSelectionStrategy):
        """
        Strategie qui priorise CriticAgent des qu'une tactique echoue.
        Delegue a la strategie parente dans les autres cas.
        """

        async def next(self, agents, history):
            # Exercice: si proof_state.error_count > 0, retourner CriticAgent
            # Exercice: sinon, deleguer a super().next(agents, history)
            if self.proof_state.error_count > 0 : 
                for agent in agents:
                    if agent.name == "CriticAgent":
                        return agent
            return await super().next(agents,history)

    print("Exercice 1: PriorityCriticStrategy - a completer")
except NameError:
    print("Exercice 1: PriorityCriticStrategy - requiere Semantic Kernel (SK_AVAILABLE=True)")


Exercice 1: PriorityCriticStrategy - a completer


## Exercice 3 : Stratégie de terminaison intelligente

### Objectif

Implementez une stratégie de terminaison **multi-critères** qui décide d'arreter la conversation d'agents de maniere plus fine que le simple compteur d'itérations.

### Contexte

La classe `ProofTerminationStrategy` (section 3.3) décide quand arreter la conversation multi-agents. Actuellement, seule la limite d'itérations est utilisee. Vous allez combiner plusieurs signaux.

### Consignes

1. Subclassez `ProofTerminationStrategy` en `SmartTerminationStrategy`
2. Ajoutez un constructeur avec `max_iterations` (defaut 20) et `max_consecutive_failures` (defaut 5)
3. Implementez `should_terminate()` avec 3 critères :
   - **Succes** : si `proof_state.final_proof` n'est pas `None`, arreter immediatement
   - **Budget** : si `proof_state.iteration >= max_iterations`, arreter
   - **Stagnation** : si `proof_state.error_count >= max_consecutive_failures`, arreter
4. Sinon, retourner `False` pour continuer

**Hint** : La signature async est `async def should_terminate(self, agents, history) -> bool`. Attributs reels de `ProofState` (section 3.1) : `final_proof` (`Optional[str]`, non-`None` si preuve trouvee), `iteration` (`int`), `error_count` (`int`, incremente par `add_tactic_attempt` a chaque tactique echouee). La strategie doit tenir une reference a l'etat : stockez `proof_state` au constructeur (`self.proof_state = proof_state`).


In [24]:
# ============================================================
# Exercice 3 : SmartTerminationStrategy
# ============================================================
# Votre objectif : implementer une strategie de terminaison
# intelligente qui combine plusieurs criteres.
#
# Rappel : ProofTerminationStrategy a la méthode :
#   async def should_terminate(self, agents, history) -> bool
#
# proof_state : état courant de la preuve (cf. ProofState, section 3.1)
# proof_state.final_proof : Optional[str] (non-None = preuve trouvee)
# proof_state.iteration : int (iteration courante)
# proof_state.error_count : int (nombre de tactiques echouees)
#
# Indice : commencez par le critere le plus simple (succes),
# puis ajoutez le budget d iterations, puis la stagnation.
# ============================================================

try:
    class SmartTerminationStrategy(ProofTerminationStrategy):
        """
        Strategie de terminaison intelligente qui combine :
        1. Succes de la preuve (arret immediat)
        2. Budget d iterations depasse
        3. Stagnation detectee (trop d echecs consécutifs)
        """

        def __init__(self, max_iterations: int = 20, max_consecutive_failures: int = 5):
            super().__init__()
            self.max_iterations = max_iterations
            self.max_consecutive_failures = max_consecutive_failures

        async def should_terminate(self, agents, history):
            """Determine si la conversation doit s arreter."""
            # TODO: Etape 1 - Si proof_state.final_proof n'est pas None -> arreter
            pass
            # TODO: Etape 2 - Si proof_state.iteration >= max_iterations -> arreter
            pass
            # TODO: Etape 3 - Si proof_state.error_count >= max_consecutive_failures -> arreter
            pass
            # TODO: Sinon -> continuer (retourner False)
            return False  # TODO etudiant

    print("Exercice 3: Strategie de terminaison intelligente - a completer")
except NameError:
    print("Exercice 3: Strategie de terminaison - requiere Semantic Kernel (SK_AVAILABLE=True)")


Exercice 3: Strategie de terminaison intelligente - a completer


## 5. Conclusion et Points Cles

### Ce que nous avons appris

#### 1. Architecture Multi-Agents pour Theorem Proving

| Composant | Rôle | Implémentation SK |
|-----------|------|-------------------|
| **ProofState** | État partage synchronise | `@dataclass` + plugins |
| **Plugins** | Fonctionnalites specialisees | `@kernel_function` |
| **Agents** | Rôles specialises (Search, Tactic, Verify...) | `ChatCompletionAgent` |
| **Orchestration** | Delegation dynamique | `AgentGroupChat` + stratégies |

#### 2. Semantic Kernel vs Implémentation Ad-Hoc (Lean-8)

| Aspect | Lean-8 (Ad-Hoc) | Lean-9 (Semantic Kernel) |
|--------|-----------------|--------------------------|
| **État** | Variables globales | `ProofState` classe |
| **Agents** | Fonctions Python | `ChatCompletionAgent` |
| **Communication** | Appels directs | Message passing |
| **Extensibilite** | Modifier le code | Ajouter plugins |
| **LLM** | OpenAI direct | Abstraction SK |

#### 3. Patterns Replicables

1. **StateManager Pattern** : Un objet central pour l'état partage
2. **Plugin Pattern** : Fonctions decorees pour l'injection de dependances
3. **Delegation Pattern** : Chaque agent designe le suivant
4. **Termination Pattern** : Critères multiples (succes, timeout, max_iter)

### Limitations et Perspectives

#### Limitations actuelles

- **Simulation trop parfaite** : Trouve les lemmes directs immediatement
- **Pas de vrai Lean** : Vérification heuristique, pas de lean4 réel
- **Base de lemmes limitee** : ~50 lemmes vs 100k+ dans Mathlib
- **Pas de backtracking** : Première tactique qui marche = solution

#### Prochaines étapes (Lean-10 LeanDojo)

1. **Integration LeanDojo** : Interaction programmatique avec Lean 4
2. **Tracing Mathlib** : Extraction des 100k+ lemmes
3. **Vérification réelle** : Feedback Lean vs heuristique
4. **Benchmarks** : MiniF2F, ProofNet, LeanBench

### Resume Final

Ce notebook a démontre comment construire un système multi-agents pour le theorem proving avec Semantic Kernel. Les patterns (StateManager, Plugin, Delegation) sont replicables pour d'autres domaines.

**Key Takeaways** :
- L'architecture compte plus que les résultats de simulation
- Semantic Kernel simplifie l'orchestration multi-agents
- Les vrais defis apparaissent avec des théorèmes sans lemmes directs
- LeanDojo (Notebook 10) permettra la vérification réelle

***

### Navigation

| Précédent | Index | Suivant |
|-----------|-------|---------|
| [Lean-8 (Agents Ad-Hoc)](Lean-08-Agentic-Proving-Python.ipynb) | [Lean-1 (Setup)](Lean-01-Setup-Lean-Python.ipynb) | [Lean-10 (LeanDojo)](Lean-10-LeanDojo.ipynb) |

***

*Notebook complète. Duree estimée: 45-55 minutes.*